# Module 1 Assignment Project

Designing a simple data product (dashboard) using real Singapore Job's Posting csv data. 

Our team is addressing a gap faced by companies entering the Singapore market: no reliable, industry-specific benchmark for talent costs or local hiring pool depth, even though headcount is typically the largest line item in an entry budget.

Our goal is to establish insights for clients, so that they can select their target industry and receive an evidence-based view of prevailing salary ranges by industry or by job title roles with data from 2023 - 2024.

### Structure

- Load and inspect initial EDA
- Remove missing values
- Clean date formats
- Clean title and text
- Remove duplicate postings
- Validate positions levels and experience
- Clean and flag unreliable salaries
- Convert hourly and annual salaries to monthly
- Parse the categories JSON
- .explode() into the long category table
- Job roles filtering from 'title_clean'
- Build analysis groupings
- Write the cleaned files
- Build the Streamlit dashboard

> The one design decision that matters is to acknowledge categories is one-to-many: a posting can sit in several industries. We keep a wide table at one row per posting for all headline numbers, and a separate exploded table at one row per posting x category for industry breakdowns.

In [1]:
import pandas as pd
import numpy as np
import re
import json

## 1. Exploratory Data Analysis

In [3]:
df = pd.read_csv('SGJobData.csv')

#Initial exploratory info

df.info()
df.describe(include='all')


<class 'pandas.DataFrame'>
RangeIndex: 1048585 entries, 0 to 1048584
Data columns (total 22 columns):
 #   Column                              Non-Null Count    Dtype  
---  ------                              --------------    -----  
 0   categories                          1044597 non-null  str    
 1   employmentTypes                     1044597 non-null  str    
 2   metadata_expiryDate                 1044597 non-null  str    
 3   metadata_isPostedOnBehalf           1048585 non-null  bool   
 4   metadata_jobPostId                  1044597 non-null  str    
 5   metadata_newPostingDate             1044597 non-null  str    
 6   metadata_originalPostingDate        1044597 non-null  str    
 7   metadata_repostCount                1048585 non-null  int64  
 8   metadata_totalNumberJobApplication  1048585 non-null  int64  
 9   metadata_totalNumberOfView          1048585 non-null  int64  
 10  minimumYearsExperience              1048585 non-null  int64  
 11  numberOfVacancies     

,categories,employmentTypes,metadata_expiryDate,metadata_isPostedOnBehalf,metadata_jobPostId,metadata_newPostingDate,metadata_originalPostingDate,metadata_repostCount,metadata_totalNumberJobApplication,metadata_totalNumberOfView,...,occupationId,positionLevels,postedCompany_name,salary_maximum,salary_minimum,salary_type,status_id,status_jobStatus,title,average_salary
count,1044597,1044597,1044597,1048585,1044597,1044597,1044597,1.048585e+06,1.048585e+06,1.048585e+06,...,0.0,1044597,1044597,1.048585e+06,1.048585e+06,1044597,1048585.0,1044597,1044597,1.048585e+06
unique,21125,8,453,2,1044597,431,603,NaN,NaN,NaN,...,NaN,9,53151,NaN,NaN,1,NaN,3,377084,NaN
top,"[{""id"":21,""category"":""Information Technology""}]",Permanent,2023-07-28,False,MCF-2023-0252866,2023-06-09,2023-07-14,NaN,NaN,NaN,...,NaN,Executive,THE SUPREME HR ADVISORY PTE. LTD.,NaN,NaN,Monthly,NaN,Open,SUPERVISOR,NaN
freq,92869,458139,4487,986717,1,4508,4029,NaN,NaN,NaN,...,NaN,253701,61638,NaN,NaN,1044597,NaN,902614,8331,NaN
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.472327e-02,2.136571e+00,2.674536e+01,...,NaN,NaN,NaN,5.723578e+03,3.815312e+03,NaN,0.0,NaN,NaN,4.769445e+03
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.822675e-01,1.062612e+01,8.262001e+01,...,NaN,NaN,NaN,5.018387e+04,3.172182e+03,NaN,0.0,NaN,NaN,2.547809e+04
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000e+00,0.000000e+00,0.000000e+00,...,NaN,NaN,NaN,0.000000e+00,0.000000e+00,NaN,0.0,NaN,NaN,0.000000e+00
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000e+00,0.000000e+00,1.000000e+00,...,NaN,NaN,NaN,3.300000e+03,2.500000e+03,NaN,0.0,NaN,NaN,2.900000e+03
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000e+00,0.000000e+00,4.000000e+00,...,NaN,NaN,NaN,4.500000e+03,3.000000e+03,NaN,0.0,NaN,NaN,3.800000e+03
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000e+00,1.000000e+00,1.700000e+01,...,NaN,NaN,NaN,6.500000e+03,4.500000e+03,NaN,0.0,NaN,NaN,5.500000e+03


### Initial information gathered

- Total of **1048585 rows** and **22 columns**
- Some have missing rows (showing 1044597 only): **3988 missing values**
- Column **'occupationID'** is totally empty (all Null values)
- **Dated columns listed as object** --> Indicating string format (need to convert)
- Column **'categories' listed as object** (looks like json, but in string format; to check)
- 'salary_minimum' and 'salary_maximum' have **outliers**

## 1.1 Removing Missing Values (NaN)

In [4]:
# Checking missing values

missing_values = df.isna().sum()

print(missing_values[missing_values > 0].sort_values(ascending=False))

occupationId                    1048585
categories                         3988
employmentTypes                    3988
metadata_expiryDate                3988
metadata_jobPostId                 3988
metadata_newPostingDate            3988
metadata_originalPostingDate       3988
positionLevels                     3988
postedCompany_name                 3988
salary_type                        3988
status_jobStatus                   3988
title                              3988
dtype: int64


Similar number of missing values. To drop these rows as it does not help in analysis (no usable data value).

In [5]:
# Entire row of occupationId is empty. Initial df.dropna() via rows will drop every single row. Hence drop via column.

# Drop empty occupationId column
df_clean = df.drop(columns=['occupationId'])

# We use jobs id column which is unique, except for the 3988 empty rows, to drop the empty jobs id rows.
df_clean = df_clean.dropna(subset=['metadata_jobPostId']).copy()

#check if rows are dropped
df_clean.isna().sum()

categories                            0
employmentTypes                       0
metadata_expiryDate                   0
metadata_isPostedOnBehalf             0
metadata_jobPostId                    0
metadata_newPostingDate               0
metadata_originalPostingDate          0
metadata_repostCount                  0
metadata_totalNumberJobApplication    0
metadata_totalNumberOfView            0
minimumYearsExperience                0
numberOfVacancies                     0
positionLevels                        0
postedCompany_name                    0
salary_maximum                        0
salary_minimum                        0
salary_type                           0
status_id                             0
status_jobStatus                      0
title                                 0
average_salary                        0
dtype: int64

## 1.2 Parsing Date type

In [6]:
# Parsing the date columns - convert date object to datetime 

date_columns = ['metadata_expiryDate', 'metadata_newPostingDate', 'metadata_originalPostingDate']

for c in date_columns:
    df_clean[c] = pd.to_datetime(df_clean[c], errors='coerce')

In [7]:
print('Check for unparseable dates:', {c: df_clean[c].isna().sum() for c in date_columns})
print('Data date range:', df_clean['metadata_originalPostingDate'].min().date(), '->',
                     df_clean['metadata_originalPostingDate'].max().date())

Check for unparseable dates: {'metadata_expiryDate': np.int64(0), 'metadata_newPostingDate': np.int64(0), 'metadata_originalPostingDate': np.int64(0)}
Data date range: 2022-10-03 -> 2024-05-29


## 1.3 Cleaning 'title' Column

'title' column was extremely messy with various forms of input since it is free text. To avoid over-normalizing the data,
Removing the following:
- Whitespaces before and after
- hashtags
- Punctuations and emojis
- Recruiter Codes at the start of entry
- Any other recruiter codes within the text line or salary

In [8]:
REGEX_RULES = [
    ('recruiter_code', r'^\s*\d{3,6}\s*[-:]\s*', ' '),          # recruiter codes at the start
    ('hashtags', r'#\w+', ' '),                                 # hashtags
    ('other_numbers', r'\b(?=[a-z0-9]*\d)[a-z0-9]{3,}\b', ' '), # any other numbers of 3 and above
    ('punction_emojis', r'[^a-z]', ' '),                        # punctuations, emojis
    ('collapse_ws', r'\s+', ' '),                               # collapse whitespace
]

lower_title = df_clean['title'].str.lower()                     # make all cell values lowercase

for name, pattern, replace in REGEX_RULES:                      # reassignment with regex, replace if pattern matches
    lower_title = lower_title.str.replace(pattern, replace, regex=True)

df_clean['title_clean'] = lower_title.str.strip()               # keep 'title' column, create new column 'title_clean' with normalized entry


# after cleaning, some rows become empty. if so, replace with original 'title'

df_clean['title_clean'] = df_clean['title_clean'].where(
    df_clean['title_clean'] != '', df_clean['title'].str.lower())

## 1.4 Removing duplicated postings

Removing any duplicated postings due to original postings expiring. 4 parameters elected to identify duplicate rows:
- Using cleaned titles: 'title_clean'
- Using company name: 'postedCompany_name'
- Using salary: 'salary_minimum' and 'salary_maximum'

In [9]:
# Duplicated postings, defined as new postings of similar job due to previous posting expiring.

key_parameters = ['title_clean', 'postedCompany_name', 'salary_minimum', 'salary_maximum']

# Sort by original posting date first, drop duplicates but keep first posting
df_clean = df_clean.sort_values('metadata_originalPostingDate').drop_duplicates(subset= key_parameters, keep='first')

print(df_clean.shape)


(629246, 22)


## 1.5 Validating Position Levels against Years of Experience

To clean for positions with contradicting position levels vs years of experience

In [10]:
# Contradictions between stated job level and required experience.
JUNIOR_LEVELS = ['Fresh/entry level', 'Junior Executive']
SENIOR_LEVELS = ['Middle Management', 'Senior Management', 'Senior Executive']

years = df_clean['minimumYearsExperience']

# Senior label but almost no experience required
too_junior_for_label = df_clean['positionLevels'].isin(SENIOR_LEVELS) & (years < 3)

# Junior label but decades of experience required
too_senior_for_label = df_clean['positionLevels'].isin(JUNIOR_LEVELS) & (years > 20)

# Implausible experience regardless of label
implausible_years = years > 40

contradictory = too_junior_for_label | too_senior_for_label

print(f'Senior label, under 3 years:  {too_junior_for_label.sum():,}')
print(f'Junior label, over 20 years:  {too_senior_for_label.sum():,}')
print(f'Experience over 40 years:     {implausible_years.sum():,}')
print(f'Total contradictory labels:   {contradictory.sum():,} '
      f'({contradictory.mean():.2%} of postings)')

# Keep the row, blank the untrustworthy value. If we drop rows, it will affect hiring pool depth.
df_clean['positionLevels_clean'] = df_clean['positionLevels'].where(~contradictory)
df_clean.loc[implausible_years, 'minimumYearsExperience'] = np.nan

print(f"\npositionLevels_clean is null for "
      f"{df_clean['positionLevels_clean'].isna().sum():,} postings "
      f"({df_clean['positionLevels_clean'].isna().mean():.1%})")
df_clean['positionLevels_clean'].value_counts(dropna=False)

Senior label, under 3 years:  16,408
Junior label, over 20 years:  28
Experience over 40 years:     19
Total contradictory labels:   16,436 (2.61% of postings)

positionLevels_clean is null for 16,436 postings (2.6%)


positionLevels_clean
Executive            152290
Junior Executive      94169
Non-executive         81278
Professional          73458
Manager               72111
Fresh/entry level     58517
Senior Executive      48582
NaN                   16436
Middle Management     16372
Senior Management     16033
Name: count, dtype: int64

## 1.6 Dealing with Salary Outliers

Setting a boundary for the monthly salary to avoid outliers, not considering the typos for hourly rates and annual rates.

In [11]:
sal = ['salary_minimum', 'salary_maximum']

SALARY_FLOOR, SALARY_CEILING = 500, 60000

# Replace any empty salary with NaN
df_clean[sal] = df_clean[sal].replace(0, np.nan)

# Rebuild without 0 salary job postings
df_clean['average_salary'] = df_clean[sal].mean(axis=1)

# Set monthly pay range to remove any typos or wrong entries
df_clean_salary = df_clean[df_clean['average_salary'].between(SALARY_FLOOR, SALARY_CEILING)].copy()

## 1.6.1 Converting hourly and annual salaries to Monthly
We want to take into account typos for hour and annual salaries to be viable data to keep integrity of the data.

#### Rules:

| Rule | Reading | Action |
|---|---|---|
| `salary_min_clean` < 100 | hourly rate | x hours x 52 / 12 (44.3h full-time, 21h part-time, MoM averages) |
| min < 0.2 x max, max > 20,500 **AND** max/12 >= min | max quoted annually | max / 12 |
| min < 0.2 x max, max <= 20,500 | range too wide to trust | both set to the raw average |
| min > 40,000 | both quoted annually | both / 12 |



In [13]:
#####SETUP####
# Work from the raw bounds; only rows with a present value get touched.
df_clean['salary_min_clean'] = df_clean['salary_minimum']
df_clean['salary_max_clean'] = df_clean['salary_maximum']

has_min = df_clean['salary_min_clean'].notna()
has_max = df_clean['salary_max_clean'].notna()
both = has_min & has_max

# For rows with unreliable small maximum or extremely high maximum: unusable, blank it rather than drop the posting
df_clean.loc[has_max & ((df_clean['salary_max_clean'] < 5) | (df_clean['salary_max_clean'] >= 1000000)),
             ['salary_min_clean', 'salary_max_clean']] = np.nan
has_min = df_clean['salary_min_clean'].notna()
has_max = df_clean['salary_max_clean'].notna()
both = has_min & has_max

# For rows if only the maximum was filled in properly: mirror it into the minimum
only_max_ok = both & (df_clean['salary_min_clean'] < 1000) & \
              (df_clean['salary_min_clean'] < 0.2 * df_clean['salary_max_clean'])
df_clean.loc[only_max_ok, 'salary_min_clean'] = df_clean.loc[only_max_ok, 'salary_max_clean']
print(f'Minimum mirrored from maximum: {only_max_ok.sum():,}')

Minimum mirrored from maximum: 1,330


In [14]:
# Changing potential hourly wages to monthly
# Monthly = hourly x weekly hours x 52 / 12.  
# MoM average hours: 44.3 full-time, 21 part-time. https://stats.mom.gov.sg

hourly = df_clean['salary_min_clean'].notna() & (df_clean['salary_min_clean'] < 100)

# A senior role quoted at an hourly-looking rate is a data error, not an
# hourly job. Flag rather than convert.
salary_level_mismatch = hourly & df_clean['positionLevels_clean'].isin(SENIOR_LEVELS)
hourly = hourly & ~salary_level_mismatch

part_time = df_clean['employmentTypes'].eq('Part Time')
hours = np.where(part_time, 21, 44.3)
factor = pd.Series(hours * 52 / 12, index=df_clean.index)

for column in ['salary_min_clean', 'salary_max_clean']:
    df_clean.loc[hourly, column] = df_clean.loc[hourly, column] * factor[hourly]

print(f'Converted from hourly:        {hourly.sum():,}')
print(f'Senior roles at hourly rates: {salary_level_mismatch.sum():,} (flagged, not converted)')

Converted from hourly:        3,361
Senior roles at hourly rates: 39 (flagged, not converted)


In [15]:
# Changing potential annual wages to monthly

# A maximum is read as annual only if it is too large to be monthly AND
# dividing it by 12 leaves a figure still at or above the minimum. 

ANNUAL_MAX_THRESHOLD = 20500    # if higher than this, a maximum may be an annual figure
BOTH_ANNUAL_THRESHOLD = 40000   # a monthly minimum this high is rare apart from senior mgt roles

both = df_clean['salary_min_clean'].notna() & df_clean['salary_max_clean'].notna()
wide_range = both & (df_clean['salary_min_clean'] < 0.2 * df_clean['salary_max_clean'])
candidate_monthly = df_clean['salary_max_clean'] / 12

# BOTH masks are computed before either is applied. 
max_annual = (wide_range
              & (df_clean['salary_max_clean'] > ANNUAL_MAX_THRESHOLD)
              & (candidate_monthly >= df_clean['salary_min_clean']))

# A row that failed the test - too wide spread.
too_wide = wide_range & ~max_annual

# Maximum quoted annually, minimum monthly
df_clean.loc[max_annual, 'salary_max_clean'] = candidate_monthly[max_annual]

# If range too wide to - collapse both ends onto the raw average
df_clean.loc[too_wide, 'salary_min_clean'] = df_clean.loc[too_wide, 'average_salary']
df_clean.loc[too_wide, 'salary_max_clean'] = df_clean.loc[too_wide, 'average_salary']

# Both ends quoted annually. Evaluated after the steps above, so a minimum only
# revealed as annual by an earlier fix is still caught.
both_annual = df_clean['salary_min_clean'].notna() & \
              (df_clean['salary_min_clean'] > BOTH_ANNUAL_THRESHOLD)
for column in ['salary_min_clean', 'salary_max_clean']:
    df_clean.loc[both_annual, column] = df_clean.loc[both_annual, column] / 12

# Junior label on a very high monthly salary is a mismatch, not a conversion
junior_high = df_clean['positionLevels_clean'].isin(JUNIOR_LEVELS) & \
              (df_clean['salary_max_clean'] >= ANNUAL_MAX_THRESHOLD)
salary_level_mismatch = salary_level_mismatch | junior_high

assert not (max_annual & too_wide).any(), 'annual rules overlap - check the masks'

# None of the three rules can invert a range: the annual reading is refused
# unless it stays above the minimum, collapsing sets both ends equal, and
# dividing both ends preserves their order. This asserts that stays true.
inverted = both & (df_clean['salary_max_clean'] < df_clean['salary_min_clean'])
assert inverted.sum() == 0, (
    f'{inverted.sum():,} postings ended up with a maximum below their minimum')

print(f'Maximum converted from annual: {max_annual.sum():,}')
print(f'Range collapsed to average:    {too_wide.sum():,}')
print(f'Both ends from annual:         {both_annual.sum():,}')
print(f'Junior label, high pay:        {junior_high.sum():,} (flagged)')
print('No inverted ranges.')

Maximum converted from annual: 148
Range collapsed to average:    287
Both ends from annual:         302
Junior label, high pay:        46 (flagged)
No inverted ranges.


In [16]:
# Final clean

# converted average, and the mismatch flag the dashboard filters
df_clean['salary_min_clean'] = df_clean['salary_min_clean'].round()
df_clean['salary_max_clean'] = df_clean['salary_max_clean'].round()
df_clean['average_salary_clean'] = (
    df_clean[['salary_min_clean', 'salary_max_clean']].mean(axis=1)
)
df_clean['salary_level_mismatch'] = salary_level_mismatch

rescued = (df_clean['average_salary_clean'].between(SALARY_FLOOR, SALARY_CEILING)
           & ~df_clean['average_salary'].between(SALARY_FLOOR, SALARY_CEILING))
print(f'Postings rescued into the usable band by conversion: {rescued.sum():,}')
print()
print(df_clean[['average_salary', 'average_salary_clean']].describe().round(0))

Postings rescued into the usable band by conversion: 3,705

       average_salary  average_salary_clean
count        629246.0              627710.0
mean           5127.0                4966.0
std           32822.0                3292.0
min               1.0                   4.0
25%            2950.0                2975.0
50%            3950.0                4000.0
75%            6000.0                6000.0
max        12666400.0               65000.0


## 1.7 Parse Categories + id into list columns
We parse the categories into list columns on df_clean, then use .explode() to build the long table.

In [17]:
# Parse safely: return [] instead of raising, so one bad row cannot kill the run.

def parse_categories(json_string):
    if pd.isna(json_string):                                    # if empty, return [] instead of None
        return []
    try:
        items = json.loads(json_string)
    except (TypeError, ValueError):                             # if json.loads fail, return [] instead of Error
        return []
    if not isinstance(items, list):                             # if not list, return []
        return []
    # de-duplicate within a posting, keep original order
    seen, out = set(), []
    for c in items:
        if not isinstance(c, dict) or c.get('category') is None:
            continue
        name = str(c['category']).strip()
        if name in seen:
            continue
        seen.add(name)
        out.append((c.get('id'), name))
    return out


parsed = df_clean['categories'].apply(parse_categories)

# Two aligned list columns: same length per row, so they explode together later
df_clean['category_id_list'] = parsed.apply(lambda rows: [i for i, _ in rows])
df_clean['category_list']    = parsed.apply(lambda rows: [n for _, n in rows])
df_clean['n_categories']     = df_clean['category_list'].str.len()

# First category only for a quick groupby 
# User-facing filter must use category_list, not this.
df_clean['main_category'] = df_clean['category_list'].str[0]

print('Rows that failed to parse or were empty:', (df_clean['n_categories'] == 0).sum())
print()
print(df_clean['n_categories'].value_counts().sort_index())

Rows that failed to parse or were empty: 0

n_categories
1    417583
2    111329
3     52309
4     22422
5     25603
Name: count, dtype: int64


In [18]:
print(f"Postings in more than one industry: {(df_clean['n_categories'] > 1).mean():.1%}")
print(f"Postings with no industry at all:   {(df_clean['n_categories'] == 0).mean():.1%}")
print(f"Total postings:                     {len(df_clean):,}")
print(f"Total posting x category pairs:     {df_clean['n_categories'].sum():,}")

df_clean[['title', 'category_list', 'main_category', 'n_categories']].head(5)

Postings in more than one industry: 33.6%
Postings with no industry at all:   0.0%
Total postings:                     629,246
Total posting x category pairs:     1,014,871


,title,category_list,main_category,n_categories
15438,Quantity Surveyor - Structural Steel,"[Building and Construction, Engineering]",Building and Construction,2
20535,Preschool Teacher (Foreigner / Local),[Education and Training],Education and Training,1
23658,Haircut Specialist,"[Customer Service, Personal Care / Beauty, Sal...",Customer Service,4
23805,Senior Purchasing Executive (Marine),[Purchasing / Merchandising],Purchasing / Merchandising,1
17692,Assistant Chef,"[F&B, General Work]",F&B,2


## 1.8 .explode() into long table

In [19]:
df_categories_exploded = (
    df_clean[['metadata_jobPostId', 'category_id_list', 'category_list']]
    .explode(['category_id_list', 'category_list'])          # pass lists to explode, unnest in parallel
    .rename(columns={'category_id_list': 'category_id',
                     'category_list': 'category_name'})
    .dropna(subset=['category_name'])                        # drop the empty-category rows i.e. any rows with no categories
    .reset_index(drop=True)
)

# Ensuring same dtypes
df_categories_exploded['category_id'] = df_categories_exploded['category_id'].astype('int64')
df_categories_exploded['category_name'] = df_categories_exploded['category_name'].astype('category')

print(f"df_clean:                {len(df_clean):,} rows (one per posting)")
print(f"df_categories_exploded:  {len(df_categories_exploded):,} rows (one per posting x category)")
df_categories_exploded.head(8)

# Assign df_categories to exploded
df_categories = df_categories_exploded

df_clean:                629,246 rows (one per posting)
df_categories_exploded:  1,014,871 rows (one per posting x category)


## 1.8.1 Overview of all categories and ids

In [20]:
category_lookup = (
    df_categories[['category_id', 'category_name']]
    .drop_duplicates()
    .sort_values('category_id')
    .reset_index(drop=True)
)
print(f'{len(category_lookup)} distinct industries')

category_counts = df_categories['category_name'].value_counts()                     # count for each categories
category_lookup['postings'] = category_lookup['category_name'].map(category_counts) # map count using category name, to new column, postings
category_lookup

43 distinct industries


,category_id,category_name,postings
0,1,Accounting / Auditing / Taxation,49027
1,2,Admin / Secretarial,70357
2,3,Advertising / Media,11947
3,4,Architecture / Interior Design,9744
4,5,Banking and Finance,40026
5,6,Building and Construction,53918
6,7,Consulting,21389
7,8,Customer Service,62119
8,9,Design,12880
9,10,Education and Training,25157


## 1.9 Job Roles from 'title_clean'
Our brief covers benchmarking "by industry" and also "by role". 
This section covers the role dimension, using 'title_clean'

In [21]:
ROLE_PATTERNS = [
    ('Data / Analytics',           r'\b(data scientist|data analyst|data engineer|business intelligence|analytics|machine learning)\b'),
    ('Software Engineering',       r'\b(software engineer|developer|programmer|full stack|front end|back end|devops|qa engineer|test engineer)\b'),
    ('IT / Infrastructure',        r'\b(it (support|executive|manager|specialist)|system(s)? (admin|engineer|analyst)|network engineer|cyber ?security|cloud engineer|helpdesk|technical support)\b'),
    ('Product / Project',          r'\b(product manager|product owner|project manager|project executive|scrum master|business analyst)\b'),
    ('Design',                     r'\b(designer|creative director|art director|graphic)\b'),
    ('Engineering (Non-Software)', r'\b(mechanical|electrical|civil|structural|process|chemical|industrial|maintenance) engineer\b'),
    ('Sales / Business Dev',       r'\b(sales|business development|account (manager|executive)|relationship manager|retail assistant|promoter)\b'),
    ('Marketing / Comms',          r'\b(marketing|brand|content|social media|communications|public relations|copywriter)\b'),
    ('Finance / Accounting',       r'\b(account(s|ant|ing)|audit|tax|finance|financial|treasury|credit|bookkeep|payroll)\b'),
    ('Human Resources',            r'\b(hr|human resource|recruit|talent acquisition|people operations)\b'),
    ('Operations / Logistics',     r'\b(operations|logistics|warehouse|supply chain|procurement|purchasing|inventory|dispatch|driver|forklift)\b'),
    ('Healthcare',                 r'\b(nurse|nursing|doctor|physician|pharmacist|therapist|clinic|medical|dental|healthcare|caregiver)\b'),
    ('Education',                  r'\b(teacher|tutor|lecturer|trainer|educator|instructor|curriculum|childcare|preschool)\b'),
    ('Customer Service',           r'\b(customer service|customer support|call cent|service crew|receptionist|concierge|guest service)\b'),
    ('Food & Beverage',            r'\b(chef|cook|barista|waiter|waitress|kitchen|f b|restaurant|bartender|pastry)\b'),
    ('Legal / Compliance',         r'\b(legal|lawyer|solicitor|paralegal|compliance|regulatory|counsel)\b'),
    ('Admin / Secretarial',        r'\b(admin|administrative|secretar|clerk|clerical|data entry|personal assistant)\b'),
    ('Construction / Trades',      r'\b(construction|site (supervisor|engineer|manager)|foreman|carpenter|plumber|electrician|welder|painter|technician)\b'),
    ('Security / Facilities',      r'\b(security (officer|guard|supervisor)|cleaner|cleaning|housekeep|facilit|janitor)\b'),
]

ROLE_REGEX = [(label, re.compile(pattern)) for label, pattern in ROLE_PATTERNS]


def match_roles(title):                      # every role family the title matches
    if not isinstance(title, str) or not title:
        return []
    return [label for label, regex in ROLE_REGEX if regex.search(title)]


titles = df_clean['title_clean'].fillna('')
df_clean['role_list']    = titles.apply(match_roles)
df_clean['primary_role'] = df_clean['role_list'].apply(
    lambda r: r[0] if r else 'Other / Unclassified')

df_clean[['title', 'title_clean', 'primary_role', 'role_list']].head(10)

,title,title_clean,primary_role,role_list
15438,Quantity Surveyor - Structural Steel,quantity surveyor structural steel,Other / Unclassified,[]
20535,Preschool Teacher (Foreigner / Local),preschool teacher foreigner local,Education,[Education]
23658,Haircut Specialist,haircut specialist,Other / Unclassified,[]
23805,Senior Purchasing Executive (Marine),senior purchasing executive marine,Operations / Logistics,[Operations / Logistics]
17692,Assistant Chef,assistant chef,Food & Beverage,[Food & Beverage]
16957,Kitchen Crew,kitchen crew,Food & Beverage,[Food & Beverage]
18642,Assistant Chef,assistant chef,Food & Beverage,[Food & Beverage]
18694,Project Manager,project manager,Product / Project,[Product / Project]
16712,Service Crew,service crew,Customer Service,[Customer Service]
18762,Kitchen Crew,kitchen crew,Food & Beverage,[Food & Beverage]


## Analysis Grouping

In [22]:
# Salary usability

df_clean['salary_reliable'] = (
    df_clean['average_salary_clean'].between(SALARY_FLOOR, SALARY_CEILING)
    & ~df_clean['salary_level_mismatch']
)

print(f"Usable salary rows (converted): {df_clean['salary_reliable'].sum():,} "
      f"({df_clean['salary_reliable'].mean():.1%})")
print(f"df_clean_salary rows (raw):     {len(df_clean_salary):,}")
print(f"Net gain from conversion:       "
      f"{df_clean['salary_reliable'].sum() - len(df_clean_salary):+,}")

# The flag must be a superset of the raw filter, minus the level mismatches.
raw_ok = df_clean['average_salary'].between(SALARY_FLOOR, SALARY_CEILING)
lost = raw_ok & ~df_clean['salary_reliable'] & ~df_clean['salary_level_mismatch']
assert lost.sum() == 0, (
    f'{lost.sum():,} postings were usable before conversion and are not now - '
    'check the thresholds in 1.5.2')
print('\nNo posting was lost by the conversion.')

# Running describe on salary_reliable to confirm no outliers
print(df_clean.loc[df_clean['salary_reliable'],"average_salary_clean"].describe().round(0))

Usable salary rows (converted): 626,968 (99.6%)
df_clean_salary rows (raw):     623,309
Net gain from conversion:       +3,659

No posting was lost by the conversion.
count    626968.0
mean       4969.0
std        3283.0
min         500.0
25%        3000.0
50%        4000.0
75%        6000.0
max       55500.0
Name: average_salary_clean, dtype: float64


In [23]:
# Experience bands
def group_experience(years):
    if pd.isna(years):
        return None
    if years <= 1:
        return '0-1 years'
    if years <= 4:
        return '2-4 years'
    if years <= 9:
        return '5-9 years'
    return '10+ years'


# Employment type, grouped into standard vs flexible
EMPLOYMENT_GROUPS = {
    'Permanent': 'Standard', 'Full Time': 'Standard', 'Contract': 'Standard',
    'Part Time': 'Flexible / Non-Standard', 'Temporary': 'Flexible / Non-Standard',
    'Freelance': 'Flexible / Non-Standard', 'Flexi-work': 'Flexible / Non-Standard',
    'Internship/Attachment': 'Internship',
}

df_clean['experience_group'] = df_clean['minimumYearsExperience'].apply(group_experience)
df_clean['employment_group'] = df_clean['employmentTypes'].map(EMPLOYMENT_GROUPS).fillna('Other')
df_clean['posting_month']    = df_clean['metadata_originalPostingDate'].dt.to_period('M').dt.to_timestamp()

missing_exp = df_clean['experience_group'].isna().mean()
print(f'Postings with no experience value: {missing_exp:.1%} '
      '(excluded from any experience-band filter)\n')

for column in ['experience_group', 'employment_group']:
    print(f'--- {column} ---')
    print(df_clean[column].value_counts(dropna=False), end='\n\n')

Postings with no experience value: 0.0% (excluded from any experience-band filter)

--- experience_group ---
experience_group
2-4 years    264010
0-1 years    208858
5-9 years    128556
10+ years     27803
NaN              19
Name: count, dtype: int64

--- employment_group ---
employment_group
Standard                   598650
Flexible / Non-Standard     26472
Internship                   4124
Name: count, dtype: int64



## Writing Files
Output for 2 csv files
- SGJobData_clean.csv
- SGJobData_categories.csv : this is the exploded table for industry breakdown

In [62]:
SEP = '|'   # csv cannot store python lists, 3 list columns joined with |

df_out = df_clean.copy()
for column in ['category_list', 'category_id_list', 'role_list']:
    df_out[column] = df_out[column].apply(
        lambda items: SEP.join(str(i) for i in items) if len(items) else '')

df_out = df_out.drop(columns=['categories'], errors='ignore')

df_out.to_csv('SGJobData_clean.csv', index=False)
df_categories.to_csv('SGJobData_categories.csv', index=False)


## Streamlit Dashboard
Overview: Client picks a target industry or role and gets prevailing salary ranges and a read on hiring pool.

In [ ]:
%%writefile app.py
"""
Singapore hiring benchmark for market entry

Purpose
-------
Help companies estimate:
1. Where hiring demand is concentrated
2. What salary budget they should plan for
3. Which sectors appear tighter to hire for

Run
---
    python -m streamlit run app.py
"""

from pathlib import Path

import altair as alt
import pandas as pd
import streamlit as st


HERE = Path(__file__).resolve().parent
CLEAN_PATH = HERE / "SGJobData_clean.csv"
SALARY_COL = "average_salary_clean"
DATE_COLS = [
    "metadata_expiryDate",
    "metadata_newPostingDate",
    "metadata_originalPostingDate",
    "posting_month",
]

SALARY_FLOOR = 500
SALARY_CEILING = 60_000
EXPERIENCE_ORDER = ["0-1 years", "2-4 years", "5-9 years", "10+ years"]


st.set_page_config(
    page_title="Singapore hiring benchmark",
    page_icon=":material/query_stats:",
    layout="wide",
    initial_sidebar_state="expanded",
)


def group_experience(years):
    if pd.isna(years):
        return None
    if years <= 1:
        return "0-1 years"
    if years <= 4:
        return "2-4 years"
    if years <= 9:
        return "5-9 years"
    return "10+ years"


def normalize_bool(series):
    if series.dtype == bool:
        return series
    return series.astype(str).str.lower().isin(["true", "1", "yes"])


@st.cache_data(show_spinner="Loading cleaned jobs data...")
def load_data():
    try:
        df = pd.read_csv(CLEAN_PATH, parse_dates=DATE_COLS)
    except ValueError:
        df = pd.read_csv(
            CLEAN_PATH,
            parse_dates=[
                "metadata_expiryDate",
                "metadata_newPostingDate",
                "metadata_originalPostingDate",
            ],
        )

    numeric_cols = [
        "metadata_totalNumberJobApplication",
        "numberOfVacancies",
        "metadata_totalNumberOfView",
        "metadata_repostCount",
        "salary_minimum",
        "salary_maximum",
        "average_salary",
        "average_salary_clean",
        "minimumYearsExperience",
    ]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    if SALARY_COL not in df.columns and "average_salary" in df.columns:
        df[SALARY_COL] = df["average_salary"]

    if "salary_reliable" in df.columns:
        df["salary_reliable"] = normalize_bool(df["salary_reliable"])
    else:
        df["salary_reliable"] = df[SALARY_COL].between(SALARY_FLOOR, SALARY_CEILING)

    if "salary_min_clean" not in df.columns and "salary_minimum" in df.columns:
        df["salary_min_clean"] = df["salary_minimum"]
    if "salary_max_clean" not in df.columns and "salary_maximum" in df.columns:
        df["salary_max_clean"] = df["salary_maximum"]

    if "posting_month" not in df.columns:
        df["posting_month"] = (
            pd.to_datetime(df["metadata_originalPostingDate"], errors="coerce")
            .dt.to_period("M")
            .dt.to_timestamp()
        )
    else:
        df["posting_month"] = pd.to_datetime(df["posting_month"], errors="coerce")

    if "experience_group" not in df.columns and "minimumYearsExperience" in df.columns:
        df["experience_group"] = df["minimumYearsExperience"].apply(group_experience)

    df["main_category"] = df["main_category"].fillna("Unknown")
    df["primary_role"] = df["primary_role"].fillna("Other / Unclassified")
    df["title_clean"] = (
        df["title_clean"]
        if "title_clean" in df.columns
        else df["title"].astype(str).str.strip().str.lower()
    )

    vacancies = df["numberOfVacancies"].where(df["numberOfVacancies"] > 0)
    views = df["metadata_totalNumberOfView"].where(df["metadata_totalNumberOfView"] > 0)

    # Candidate-response metrics. These are more informative than repost counts
    # for this dataset because repost values are overwhelmingly zero-heavy.
    df["applications_per_vacancy"] = df["metadata_totalNumberJobApplication"] / vacancies
    df["application_rate"] = df["metadata_totalNumberJobApplication"] / views
    df["role_classified"] = df["primary_role"].ne("Other / Unclassified")

    return df


def apply_filters(frame, filters):
    dff = frame.copy()

    if filters["categories"]:
        dff = dff[dff["main_category"].isin(filters["categories"])]
    if filters["roles"]:
        dff = dff[dff["primary_role"].isin(filters["roles"])]
    if filters["positions"]:
        dff = dff[dff["positionLevels"].isin(filters["positions"])]
    if filters["employment"]:
        dff = dff[dff["employmentTypes"].isin(filters["employment"])]
    if filters["experience"]:
        dff = dff[dff["experience_group"].isin(filters["experience"])]
    if filters["title_query"]:
        dff = dff[
            dff["title_clean"].str.contains(filters["title_query"], case=False, na=False)
        ]

    start = pd.Timestamp(filters["date_range"][0])
    end = pd.Timestamp(filters["date_range"][1]) + pd.Timedelta(days=1)
    dff = dff[dff["metadata_originalPostingDate"].between(start, end)]
    return dff


def sector_benchmarks(frame, salary_frame, min_postings):
    base = (
        frame.groupby("main_category")
        .agg(
            postings=("metadata_jobPostId", "nunique"),
            mean_applications_per_vacancy=("applications_per_vacancy", "mean"),
            mean_application_rate=("application_rate", "mean"),
        )
        .reset_index()
    )

    salary = (
        salary_frame
        .groupby("main_category")[SALARY_COL]
        .median()
        .rename("median_salary")
        .reset_index()
    )

    bench = base.merge(salary, on="main_category", how="left")
    bench = bench[bench["postings"] >= min_postings].copy()
    bench = bench.dropna(
        subset=["mean_applications_per_vacancy", "mean_application_rate", "median_salary"]
    )

    if bench.empty:
        return bench

    # Higher score = tighter market for employers:
    # lower candidate response + higher pay requirement.
    bench["tightness_score"] = (
        (1 - bench["mean_applications_per_vacancy"].rank(pct=True)) * 0.45
        + (1 - bench["mean_application_rate"].rank(pct=True)) * 0.35
        + bench["median_salary"].rank(pct=True) * 0.20
    )

    bench["tightness_label"] = pd.cut(
        bench["tightness_score"],
        bins=[-0.01, 0.25, 0.50, 0.75, 1.01],
        labels=["Easier", "Balanced", "Tighter", "Tightest"],
    )
    return bench.sort_values("tightness_score", ascending=False)


def role_response_benchmarks(frame, min_postings):
    role = (
        frame[frame["role_classified"]]
        .groupby("primary_role")
        .agg(
            postings=("metadata_jobPostId", "nunique"),
            mean_applications_per_vacancy=("applications_per_vacancy", "mean"),
            mean_application_rate=("application_rate", "mean"),
        )
        .reset_index()
    )
    role = role[role["postings"] >= min_postings].copy()
    return role.sort_values("mean_applications_per_vacancy", ascending=True)


def seniority_response_benchmarks(frame, min_postings=25):
    bench = (
        frame.groupby("experience_group")
        .agg(
            postings=("metadata_jobPostId", "nunique"),
            mean_applications_per_vacancy=("applications_per_vacancy", "mean"),
            mean_application_rate=("application_rate", "mean"),
        )
        .reset_index()
    )
    bench = bench[bench["postings"] >= min_postings].copy()
    bench["experience_group"] = pd.Categorical(
        bench["experience_group"], categories=EXPERIENCE_ORDER, ordered=True
    )
    return bench.sort_values("experience_group")


def experience_pay_story(salary_exp):
    if salary_exp.empty or len(salary_exp) < 2:
        return "Experience-band pay and demand"

    exp_data = salary_exp.dropna().copy()
    if exp_data.empty:
        return "Experience-band pay and demand"

    busiest_band = exp_data.sort_values("postings", ascending=False).iloc[0]
    ordered = exp_data.set_index("experience_group").reindex(EXPERIENCE_ORDER).dropna().reset_index()
    if len(ordered) < 2:
        return f"Most demand sits at {busiest_band['experience_group']}"

    ordered["prev_salary"] = ordered["median_salary"].shift(1)
    ordered["pay_jump"] = ordered["median_salary"] - ordered["prev_salary"]
    jumps = ordered.dropna(subset=["pay_jump"])
    if jumps.empty:
        return f"Most demand sits at {busiest_band['experience_group']}"

    biggest_jump = jumps.sort_values("pay_jump", ascending=False).iloc[0]
    prev_band = ordered.loc[ordered.index[ordered["experience_group"] == biggest_jump["experience_group"]][0] - 1, "experience_group"]
    return (
        f"Most demand sits at {busiest_band['experience_group']}, while pay jumps {money_md(biggest_jump['pay_jump'])} "
        f"from {prev_band} to {biggest_jump['experience_group']}"
    )


def money(value):
    if pd.isna(value):
        return "N/A"
    return f"S${value:,.0f}"


def money_md(value):
    if pd.isna(value):
        return "N/A"
    return f"S\\${value:,.0f}"


def pct_text(value):
    if pd.isna(value):
        return "N/A"
    return f"{value:.1%}"


def build_scope_label(filters):
    if filters["title_query"]:
        base = f'"{filters["title_query"]}" roles'
    elif len(filters["roles"]) == 1:
        base = f'{filters["roles"][0]} roles'
    elif len(filters["categories"]) == 1:
        base = f'roles in {filters["categories"][0]}'
    else:
        base = "all roles in view"

    if filters["title_query"] and len(filters["categories"]) == 1:
        base += f' in {filters["categories"][0]}'
    elif filters["categories"] and len(filters["categories"]) > 1:
        base += f' across {len(filters["categories"])} selected sectors'

    return base


def filters_are_active(filters, min_date, max_date):
    start, end = filters["date_range"]
    if pd.Timestamp(start).date() != min_date or pd.Timestamp(end).date() != max_date:
        return True
    return any(
        [
            filters["categories"],
            filters["roles"],
            filters["positions"],
            filters["employment"],
            filters["experience"],
            filters["title_query"],
        ]
    )


def build_filter_badges(filters, min_date, max_date):
    badges = []
    for category in filters["categories"][:4]:
        badges.append(f":blue-badge[Sector: {category}]")
    if len(filters["categories"]) > 4:
        badges.append(f":blue-badge[+{len(filters['categories']) - 4} more sectors]")

    for role in filters["roles"][:3]:
        badges.append(f":green-badge[Job family: {role}]")
    if len(filters["roles"]) > 3:
        badges.append(f":green-badge[+{len(filters['roles']) - 3} more job families]")

    if filters["title_query"]:
        badges.append(f":orange-badge[Title contains: {filters['title_query']}]")

    if filters["employment"]:
        if len(filters["employment"]) == 1:
            badges.append(f":violet-badge[Employment: {filters['employment'][0]}]")
        else:
            badges.append(f":violet-badge[{len(filters['employment'])} employment types]")
    if filters["positions"]:
        if len(filters["positions"]) == 1:
            badges.append(f":gray-badge[Level: {filters['positions'][0]}]")
        else:
            badges.append(f":gray-badge[{len(filters['positions'])} position levels]")
    if filters["experience"]:
        if len(filters["experience"]) == 1:
            badges.append(f":green-badge[Experience: {filters['experience'][0]}]")
        else:
            badges.append(f":green-badge[{len(filters['experience'])} experience bands]")

    start, end = filters["date_range"]
    if pd.Timestamp(start).date() != min_date or pd.Timestamp(end).date() != max_date:
        badges.append(
            f":blue-badge[Dates: {pd.Timestamp(start):%b %Y} to {pd.Timestamp(end):%b %Y}]"
        )

    if not badges:
        badges.append(":blue-badge[Whole market selection]")
    return " ".join(badges)


def scope_salary_change(salary_frame):
    if salary_frame.empty:
        return None
    monthly = (
        salary_frame.groupby("posting_month")[SALARY_COL]
        .median()
        .dropna()
        .sort_index()
    )
    if len(monthly) < 2:
        return None
    first_month = monthly.index[0]
    last_month = monthly.index[-1]
    first_value = monthly.iloc[0]
    last_value = monthly.iloc[-1]
    pct_change = None if first_value in [0, None] or pd.isna(first_value) else (last_value - first_value) / first_value
    return {
        "first_month": first_month,
        "last_month": last_month,
        "first_value": first_value,
        "last_value": last_value,
        "pct_change": pct_change,
    }


def benchmark_bar_chart(selected_label, selected_value, market_value, x_title, height=280):
    max_value = max(selected_value, market_value) * 1.35 if pd.notna(selected_value) and pd.notna(market_value) else 1
    benchmark_df = pd.DataFrame(
        {
            "group": ["Benchmark"],
            "selected_value": [selected_value],
            "market_value": [market_value],
            "selected_label": [selected_label],
        }
    )
    selected_text = pd.DataFrame(
        {
            "group": ["Benchmark"],
            "value": [selected_value],
            "text": [selected_label],
        }
    )
    market_text = pd.DataFrame(
        {
            "group": ["Benchmark"],
            "value": [market_value],
            "text": ["Wider market"],
        }
    )
    return (
        alt.layer(
            alt.Chart(benchmark_df)
            .mark_bar(color="#c2410c", cornerRadiusEnd=6, size=28)
            .encode(
                x=alt.X(
                    "selected_value:Q",
                    title=x_title,
                    scale=alt.Scale(domain=[0, max_value]),
                ),
                y=alt.Y("group:N", title=None, axis=None),
                tooltip=[
                    alt.Tooltip("selected_label:N", title="Selected scope"),
                    alt.Tooltip("selected_value:Q", title=x_title),
                    alt.Tooltip("market_value:Q", title="Wider market"),
                ],
            ),
            alt.Chart(benchmark_df)
            .mark_rule(color="#0f172a", strokeWidth=3)
            .encode(x="market_value:Q"),
            alt.Chart(selected_text)
            .mark_text(align="left", dx=6, dy=-18, fontSize=11, color="#9a3412")
            .encode(x="value:Q", y=alt.Y("group:N", title=None, axis=None), text="text:N"),
            alt.Chart(market_text)
            .mark_text(align="center", dy=22, fontSize=11, color="#334155")
            .encode(x="value:Q", y=alt.Y("group:N", title=None, axis=None), text="text:N"),
        ).properties(height=height)
    )


def benchmark_title(selected_label, selected_value, market_value, metric_name, higher_is_harder=True, money_metric=False):
    if pd.isna(selected_value) or pd.isna(market_value):
        return f"{selected_label} compared with the whole market"
    diff = selected_value - market_value
    if market_value == 0:
        pct = None
    else:
        pct = diff / market_value

    if money_metric:
        if pct is None:
            return f"{selected_label} pays {money_md(selected_value)}"
        direction = "above" if diff > 0 else "below"
        return f"{selected_label} pays {abs(pct):.1%} {direction} the whole-market median"

    if metric_name == "applications_per_vacancy":
        if higher_is_harder:
            return f"{selected_label} gets {selected_value:.2f} applications per vacancy vs {market_value:.2f} market-wide"
        if selected_value < market_value:
            return f"{selected_label} attracts fewer applicants per vacancy than the whole market"
        return f"{selected_label} attracts more applicants per vacancy than the whole market"

    if metric_name == "application_rate":
        return f"{selected_label} converts {selected_value:.1%} of views into applications vs {market_value:.1%} market-wide"

    return f"{selected_label} compared with the whole market"


def share_donut_chart(selected_count, market_count, selected_label):
    other_count = max(market_count - selected_count, 0)
    share_df = pd.DataFrame(
        {
            "segment": [selected_label, "Rest of market"],
            "count": [selected_count, other_count],
        }
    )
    share_df["color_group"] = share_df["segment"].apply(
        lambda x: "Selected" if x == selected_label else "Other"
    )
    return (
        alt.Chart(share_df)
        .mark_arc(innerRadius=70, outerRadius=110)
        .encode(
            theta=alt.Theta("count:Q"),
            color=alt.Color(
                "color_group:N",
                scale=alt.Scale(
                    domain=["Selected", "Other"],
                    range=["#0f766e", "#e2e8f0"],
                ),
                legend=None,
            ),
            tooltip=["segment", "count"],
        )
        .properties(height=260)
    )


def experience_mix_comparison(selected_df, market_df):
    selected = (
        selected_df.groupby("experience_group")["metadata_jobPostId"]
        .nunique()
        .reindex(EXPERIENCE_ORDER)
        .fillna(0)
    )
    market = (
        market_df.groupby("experience_group")["metadata_jobPostId"]
        .nunique()
        .reindex(EXPERIENCE_ORDER)
        .fillna(0)
    )
    if selected.sum() == 0 or market.sum() == 0:
        return pd.DataFrame(columns=["experience_group", "share", "scope"])
    result = pd.DataFrame(
        {
            "experience_group": EXPERIENCE_ORDER * 2,
            "share": list((selected / selected.sum()).values) + list((market / market.sum()).values),
            "scope": ["Selected scope"] * len(EXPERIENCE_ORDER) + ["Whole market"] * len(EXPERIENCE_ORDER),
        }
    )
    return result


def experience_mix_title(comparison_df):
    if comparison_df.empty:
        return "Experience mix compared with the whole market"
    wide = comparison_df.pivot(index="experience_group", columns="scope", values="share").fillna(0)
    wide["diff"] = wide["Selected scope"] - wide["Whole market"]
    top_band = wide["diff"].abs().idxmax()
    diff = wide.loc[top_band, "diff"]
    selected_share = wide.loc[top_band, "Selected scope"]
    market_share = wide.loc[top_band, "Whole market"]
    if abs(diff) < 0.03:
        return "Experience mix is close to the whole market"
    if diff > 0:
        return f"{top_band} roles make up {selected_share:.0%} of selected postings vs {market_share:.0%} market-wide"
    return f"{top_band} roles make up {selected_share:.0%} of selected postings vs {market_share:.0%} market-wide"


def experience_mix_dumbbell_chart(comparison_df):
    if comparison_df.empty:
        return alt.Chart(pd.DataFrame({"x": [], "y": []})).mark_point()

    wide = (
        comparison_df.pivot(index="experience_group", columns="scope", values="share")
        .reindex(EXPERIENCE_ORDER)
        .reset_index()
    )
    rule_df = wide.rename(
        columns={"Selected scope": "selected_share", "Whole market": "market_share"}
    )
    point_df = comparison_df.copy()

    return alt.layer(
        alt.Chart(rule_df)
        .mark_rule(strokeWidth=3, color="#cbd5e1")
        .encode(
            y=alt.Y("experience_group:N", sort=EXPERIENCE_ORDER, title=None),
            x=alt.X("market_share:Q", title="Share of postings", axis=alt.Axis(format="%")),
            x2="selected_share:Q",
        ),
        alt.Chart(point_df)
        .mark_circle(size=180)
        .encode(
            y=alt.Y("experience_group:N", sort=EXPERIENCE_ORDER, title=None),
            x=alt.X("share:Q", title="Share of postings", axis=alt.Axis(format="%")),
            color=alt.Color(
                "scope:N",
                scale=alt.Scale(
                    domain=["Selected scope", "Whole market"],
                    range=["#c2410c", "#94a3b8"],
                ),
                legend=alt.Legend(title=None),
            ),
            tooltip=["experience_group", "scope", alt.Tooltip("share:Q", format=".1%")],
        ),
    ).properties(height=320)


def sector_focus_text(filtered_df):
    sector_counts = (
        filtered_df.groupby("main_category")["metadata_jobPostId"]
        .nunique()
        .sort_values(ascending=False)
    )
    if sector_counts.empty:
        return "N/A", 0.0
    top_sector = sector_counts.index[0]
    top_share = sector_counts.iloc[0] / sector_counts.sum()
    return top_sector, top_share


def job_family_focus_text(filtered_df):
    role_counts = (
        filtered_df[filtered_df["role_classified"]]
        .groupby("primary_role")["metadata_jobPostId"]
        .nunique()
        .sort_values(ascending=False)
    )
    if role_counts.empty:
        return "N/A", 0.0
    top_role = role_counts.index[0]
    top_share = role_counts.iloc[0] / role_counts.sum()
    return top_role, top_share


def response_vs_market_label(mean_apv, mean_rate, market_apv, market_rate):
    if mean_apv >= market_apv * 1.15 and mean_rate >= market_rate * 1.10:
        return "stronger than the wider market"
    if mean_apv <= market_apv * 0.85 and mean_rate <= market_rate * 0.90:
        return "weaker than the wider market"
    return "broadly in line with the wider market"


def selected_sector_title(row, bench):
    salary_mid = bench["median_salary"].median()
    apv_mid = bench["mean_applications_per_vacancy"].median()
    rate_mid = bench["mean_application_rate"].median()

    if row["median_salary"] >= salary_mid and row["mean_applications_per_vacancy"] < apv_mid:
        return f'{row["main_category"]} pays above peer median but draws fewer applicants'
    if row["median_salary"] < salary_mid and row["mean_applications_per_vacancy"] < apv_mid:
        return f'{row["main_category"]} draws fewer applicants even at lower pay'
    if row["median_salary"] >= salary_mid and row["mean_application_rate"] >= rate_mid:
        return f'{row["main_category"]} pays above peer median and still attracts interest'
    return f'{row["main_category"]} sits in the easier half of the current comparison set'


def ordinal_text(n):
    if pd.isna(n):
        return "N/A"
    n = int(n)
    if 10 <= n % 100 <= 20:
        suffix = "th"
    else:
        suffix = {1: "st", 2: "nd", 3: "rd"}.get(n % 10, "th")
    return f"{n}{suffix}"


def peer_sector_comparison_chart(
    market_bench,
    selected_label,
    selected_value,
    metric_col,
    metric_title,
    highlight_sector=None,
    top_n=8,
    ascending=False,
):
    if market_bench.empty or pd.isna(selected_value):
        return alt.Chart(pd.DataFrame({"x": [], "y": []})).mark_point()

    chart_data = market_bench.sort_values("postings", ascending=False).head(top_n).copy()
    chart_data["label"] = chart_data["main_category"]
    chart_data["group"] = "Peer sectors"

    if highlight_sector and highlight_sector in market_bench["main_category"].values:
        selected_row = market_bench[market_bench["main_category"] == highlight_sector].copy()
        selected_row["label"] = selected_row["main_category"]
        selected_row["group"] = "Selected scope"
        chart_data = pd.concat([chart_data, selected_row], ignore_index=True)
        chart_data = chart_data.drop_duplicates(subset=["label"], keep="last")
    else:
        selected_row = pd.DataFrame(
            {
                "main_category": [selected_label],
                "label": [selected_label],
                metric_col: [selected_value],
                "group": ["Selected scope"],
                "postings": [pd.NA],
            }
        )
        chart_data = pd.concat([chart_data, selected_row], ignore_index=True)
        chart_data = chart_data.drop_duplicates(subset=["label"], keep="last")

    chart_data = chart_data.sort_values(metric_col, ascending=ascending).copy()
    sort_order = chart_data["label"].tolist()
    benchmark_value = market_bench[metric_col].median()
    benchmark_df = pd.DataFrame({"benchmark": [benchmark_value]})

    return alt.layer(
        alt.Chart(benchmark_df)
        .mark_rule(strokeDash=[6, 4], color="#0f172a")
        .encode(x=alt.X("benchmark:Q", title=metric_title)),
        alt.Chart(chart_data)
        .mark_circle(size=170)
        .encode(
            x=alt.X(f"{metric_col}:Q", title=metric_title),
            y=alt.Y("label:N", sort=sort_order, title=None),
            color=alt.Color(
                "group:N",
                scale=alt.Scale(
                    domain=["Selected scope", "Peer sectors"],
                    range=["#c2410c", "#cbd5e1"],
                ),
                legend=alt.Legend(title=None),
            ),
            tooltip=[
                alt.Tooltip("label:N", title="Sector"),
                alt.Tooltip(f"{metric_col}:Q", title=metric_title),
                alt.Tooltip("postings:Q", title="Postings"),
            ],
        ),
    ).properties(height=320)


def scope_tightness_score(selected_salary, selected_apv, selected_rate, market_bench):
    if market_bench.empty:
        return None
    if any(pd.isna(v) for v in [selected_salary, selected_apv, selected_rate]):
        return None

    pay_pct = (market_bench["median_salary"] <= selected_salary).mean()
    apv_pct = (market_bench["mean_applications_per_vacancy"] <= selected_apv).mean()
    rate_pct = (market_bench["mean_application_rate"] <= selected_rate).mean()

    return (1 - apv_pct) * 0.45 + (1 - rate_pct) * 0.35 + pay_pct * 0.20


def posting_trend_story(monthly_df, label):
    if monthly_df.empty or monthly_df["postings"].sum() == 0:
        return f"{label} posting trend"

    monthly_df = monthly_df.sort_values("posting_month").copy()
    latest_month = monthly_df["posting_month"].iloc[-1]
    latest_value = monthly_df["postings"].iloc[-1]
    year_ago_month = latest_month - pd.DateOffset(years=1)
    year_ago_match = monthly_df[monthly_df["posting_month"] == year_ago_month]

    if not year_ago_match.empty and year_ago_match["postings"].iloc[0] > 0:
        year_ago_value = year_ago_match["postings"].iloc[0]
        pct_change = (latest_value - year_ago_value) / year_ago_value
        direction = "above" if pct_change >= 0 else "below"
        return (
            f"{label} postings in {latest_month:%b %Y} are {abs(pct_change):.0%} "
            f"{direction} {year_ago_month:%b %Y}"
        )

    first_month = monthly_df["posting_month"].iloc[0]
    first_value = monthly_df["postings"].iloc[0]
    if first_value > 0:
        pct_change = (latest_value - first_value) / first_value
        direction = "above" if pct_change >= 0 else "below"
        return (
            f"{label} postings in {latest_month:%b %Y} are {abs(pct_change):.0%} "
            f"{direction} {first_month:%b %Y}"
        )

    return f"{label} posting trend"


def build_peer_trend_data(market_view_df, filtered_df, selected_label, selected_categories, top_n=8):
    peer_counts = (
        market_view_df.groupby("main_category")["metadata_jobPostId"]
        .nunique()
        .sort_values(ascending=False)
    )
    peer_names = [name for name in peer_counts.index if name not in selected_categories][:top_n]

    selected_monthly = (
        filtered_df.groupby("posting_month")["metadata_jobPostId"]
        .nunique()
        .reset_index(name="postings")
    )
    selected_monthly["series_label"] = selected_label
    selected_monthly["highlight_group"] = "Selected scope"

    peer_monthly = (
        market_view_df[market_view_df["main_category"].isin(peer_names)]
        .groupby(["posting_month", "main_category"])["metadata_jobPostId"]
        .nunique()
        .reset_index(name="postings")
        .rename(columns={"main_category": "series_label"})
    )
    peer_monthly["highlight_group"] = "Peer sectors"

    combined = pd.concat([selected_monthly, peer_monthly], ignore_index=True)
    return combined.sort_values(["highlight_group", "series_label", "posting_month"])


def build_sector_salary_trend_data(
    market_salary_frame,
    filtered_salary_frame,
    selected_label,
    selected_categories,
    experience_band,
    top_n=8,
):
    band_market = market_salary_frame[
        market_salary_frame["experience_group"] == experience_band
    ].copy()
    band_selected = filtered_salary_frame[
        filtered_salary_frame["experience_group"] == experience_band
    ].copy()

    peer_counts = (
        band_market.groupby("main_category")["metadata_jobPostId"]
        .nunique()
        .sort_values(ascending=False)
    )
    peer_names = [name for name in peer_counts.index if name not in selected_categories][:top_n]

    selected_monthly = (
        band_selected.groupby("posting_month")[SALARY_COL]
        .median()
        .reset_index(name="median_salary")
    )
    selected_monthly["series_label"] = selected_label
    selected_monthly["highlight_group"] = "Selected scope"

    peer_monthly = (
        band_market[band_market["main_category"].isin(peer_names)]
        .groupby(["posting_month", "main_category"])[SALARY_COL]
        .median()
        .reset_index(name="median_salary")
        .rename(columns={"main_category": "series_label"})
    )
    peer_monthly["highlight_group"] = "Peer sectors"

    median_monthly = (
        band_market.groupby("posting_month")[SALARY_COL]
        .median()
        .reset_index(name="median_salary")
    )
    median_monthly["series_label"] = "Sector median"
    median_monthly["highlight_group"] = "Median benchmark"

    combined = pd.concat([selected_monthly, peer_monthly, median_monthly], ignore_index=True)
    combined["line_size"] = combined["highlight_group"].map(
        {"Selected scope": 4, "Median benchmark": 3, "Peer sectors": 2}
    )
    combined["line_opacity"] = combined["highlight_group"].map(
        {"Selected scope": 1.0, "Median benchmark": 0.9, "Peer sectors": 0.65}
    )
    return combined.sort_values(["highlight_group", "series_label", "posting_month"])


def band_salary_changes(salary_trend):
    trend = salary_trend.dropna().copy()
    if trend.empty:
        return []

    changes = []
    for band, band_df in trend.groupby("experience_group"):
        band_df = band_df.sort_values("posting_month")
        if len(band_df) < 2:
            continue
        first = band_df["median_salary"].iloc[0]
        last = band_df["median_salary"].iloc[-1]
        if pd.notna(first) and first > 0 and pd.notna(last):
            pct_change = (last - first) / first
            changes.append(
                {
                    "band": band,
                    "pct_change": pct_change,
                    "first": first,
                    "last": last,
                }
            )
    return changes


def salary_trend_story(salary_change, salary_trend):
    changes = band_salary_changes(salary_trend)
    if salary_change is None:
        return (
            "Salary trend by experience band",
            "No salary trend is available for this view.",
        )

    overall_change = salary_change["pct_change"]
    if overall_change is None:
        return (
            "Salary trend by experience band",
            "Overall salary movement could not be calculated for this view.",
        )

    if not changes:
        return (
            f"Overall median pay changed {overall_change:+.1%} across the selected period",
            f"Across the selected period, overall median pay moved from **{money_md(salary_change['first_value'])}** in **{salary_change['first_month']:%b %Y}** to **{money_md(salary_change['last_value'])}** in **{salary_change['last_month']:%b %Y}**, a change of **{overall_change:+.1%}**.",
        )

    biggest = max(changes, key=lambda x: abs(x["pct_change"]))
    if abs(overall_change) < 0.05:
        return (
            "Overall pay is broadly flat across the selected period",
            f"Overall median pay moved from **{money_md(salary_change['first_value'])}** in **{salary_change['first_month']:%b %Y}** to **{money_md(salary_change['last_value'])}** in **{salary_change['last_month']:%b %Y}** (**{overall_change:+.1%}**). The largest band-level move was **{biggest['band']}**, which changed from **{money_md(biggest['first'])}** to **{money_md(biggest['last'])}** (**{biggest['pct_change']:+.1%}**).",
        )

    direction = "rose" if overall_change > 0 else "fell"
    return (
        f"Overall median pay {direction} {abs(overall_change):.1%} across the selected period",
        f"Overall median pay moved from **{money_md(salary_change['first_value'])}** in **{salary_change['first_month']:%b %Y}** to **{money_md(salary_change['last_value'])}** in **{salary_change['last_month']:%b %Y}** (**{overall_change:+.1%}**). The largest band-level move was **{biggest['band']}**, from **{money_md(biggest['first'])}** to **{money_md(biggest['last'])}** (**{biggest['pct_change']:+.1%}**).",
    )


def top_highlight_chart(data, category_col, value_col, highlight_value, x_title, height=360):
    chart_data = data.copy()
    chart_data["highlight_group"] = chart_data[category_col].apply(
        lambda x: "Highlighted" if x == highlight_value else "Other"
    )
    return (
        alt.Chart(chart_data)
        .mark_bar()
        .encode(
            x=alt.X(f"{value_col}:Q", title=x_title),
            y=alt.Y(f"{category_col}:N", sort="-x", title=None),
            color=alt.Color(
                "highlight_group:N",
                scale=alt.Scale(
                    domain=["Highlighted", "Other"],
                    range=["#0f766e", "#cbd5e1"],
                ),
                legend=None,
            ),
            tooltip=[category_col, value_col],
        )
        .properties(height=height)
    )


def tightness_rank_chart(
    bench,
    highlight_sector=None,
    selected_score=None,
    selected_label=None,
    top_n=8,
    height=360,
):
    chart_data = bench.sort_values("tightness_score", ascending=False).head(top_n).copy()
    if highlight_sector and highlight_sector in bench["main_category"].values:
        selected_row = bench[bench["main_category"] == highlight_sector].copy()
        chart_data = pd.concat([chart_data, selected_row], ignore_index=True)
        chart_data = chart_data.drop_duplicates(subset=["main_category"], keep="last")
    elif selected_label and selected_score is not None:
        selected_row = pd.DataFrame(
            {
                "main_category": [selected_label],
                "tightness_score": [selected_score],
                "median_salary": [pd.NA],
                "mean_applications_per_vacancy": [pd.NA],
                "mean_application_rate": [pd.NA],
                "postings": [pd.NA],
            }
        )
        chart_data = pd.concat([chart_data, selected_row], ignore_index=True)
        chart_data = chart_data.drop_duplicates(subset=["main_category"], keep="last")

    highlight_name = highlight_sector if highlight_sector else selected_label
    if highlight_name is None and not chart_data.empty:
        highlight_name = chart_data.iloc[0]["main_category"]
    chart_data = chart_data.sort_values("tightness_score", ascending=False).copy()
    chart_data["highlight_group"] = chart_data["main_category"].apply(
        lambda x: "Highlighted" if x == highlight_name else "Other"
    )
    return (
        alt.Chart(chart_data)
        .mark_bar(cornerRadiusEnd=5)
        .encode(
            y=alt.Y(
                "main_category:N",
                title=None,
                sort=chart_data["main_category"].tolist(),
            ),
            x=alt.X("tightness_score:Q", title="Hiring tightness score"),
            color=alt.Color(
                "highlight_group:N",
                scale=alt.Scale(
                    domain=["Highlighted", "Other"],
                    range=["#b91c1c", "#cbd5e1"],
                ),
                legend=None,
            ),
            tooltip=[
                "main_category",
                alt.Tooltip("tightness_score:Q", format=".2f"),
                "median_salary",
                "mean_applications_per_vacancy",
                "mean_application_rate",
                "postings",
            ],
        )
        .properties(height=height)
    )


def salary_by_experience(frame):
    result = (
        frame[frame["salary_reliable"]]
        .groupby("experience_group")[SALARY_COL]
        .agg(median_salary="median", postings="size")
        .reset_index()
    )
    result["experience_group"] = pd.Categorical(
        result["experience_group"], categories=EXPERIENCE_ORDER, ordered=True
    )
    return result.sort_values("experience_group")


def salary_trend_by_experience(frame):
    result = (
        frame[frame["salary_reliable"]]
        .groupby(["posting_month", "experience_group"])[SALARY_COL]
        .median()
        .reset_index(name="median_salary")
    )
    result["experience_group"] = pd.Categorical(
        result["experience_group"], categories=EXPERIENCE_ORDER, ordered=True
    )
    return result.sort_values(["experience_group", "posting_month"])


df = load_data()

min_date = df["metadata_originalPostingDate"].min().date()
max_date = df["metadata_originalPostingDate"].max().date()

category_options = sorted(df["main_category"].dropna().unique().tolist())
role_options = sorted(
    df.loc[df["role_classified"], "primary_role"].dropna().unique().tolist()
)
employment_options = sorted(df["employmentTypes"].dropna().unique().tolist())

with st.sidebar:
    st.header("Filters")

    if st.button("Reset filters"):
        st.session_state.clear()
        st.rerun()

    selected_dates = st.date_input(
        "Posting date range",
        value=(min_date, max_date),
        min_value=min_date,
        max_value=max_date,
    )

    selected_categories = st.multiselect("Sector", category_options)
    selected_roles = st.multiselect("Role family", role_options)
    selected_employment = st.multiselect("Employment type", employment_options)
    selected_experience = st.multiselect("Experience band", EXPERIENCE_ORDER)
    title_query = st.text_input("Title contains", placeholder="engineer, analyst, nurse")

    salary_only = st.checkbox(
        "Use salary-clean rows only",
        value=True,
        help=f"Limits salary analysis to rows between S${SALARY_FLOOR:,} and S${SALARY_CEILING:,}.",
    )
    min_n = st.slider(
        "Minimum postings per benchmark group",
        min_value=25,
        max_value=1000,
        value=100,
        step=25,
    )

filters = {
    "date_range": selected_dates if len(selected_dates) == 2 else (min_date, max_date),
    "categories": selected_categories,
    "roles": selected_roles,
    "positions": [],
    "employment": selected_employment,
    "experience": selected_experience,
    "title_query": title_query.strip(),
    "salary_only": salary_only,
}

filtered_df = apply_filters(df, filters)
market_filters = {
    "date_range": filters["date_range"],
    "categories": [],
    "roles": [],
    "positions": [],
    "employment": [],
    "experience": [],
    "title_query": "",
    "salary_only": salary_only,
}
market_view_df = apply_filters(df, market_filters)
salary_frame = (
    filtered_df[filtered_df["salary_reliable"]].copy()
    if salary_only
    else filtered_df[filtered_df[SALARY_COL].notna()].copy()
)
market_salary_frame = (
    market_view_df[market_view_df["salary_reliable"]].copy()
    if salary_only
    else market_view_df[market_view_df[SALARY_COL].notna()].copy()
)

market_bench = sector_benchmarks(market_view_df, market_salary_frame, min_n)
role_bench = role_response_benchmarks(filtered_df, min_n)
seniority_bench = seniority_response_benchmarks(filtered_df, min_postings=25)
salary_exp = salary_by_experience(salary_frame)
salary_trend = salary_trend_by_experience(salary_frame)
scope_label = build_scope_label(filters)
filters_active = filters_are_active(filters, min_date, max_date)
filter_badges = build_filter_badges(filters, min_date, max_date)

st.title("Singapore hiring benchmark for market entry")
st.caption(
    "Leave sector and job-family filters blank to see the full market in the selected date range. "
    "Add a sector, job family, title, employment type, level, or experience filter to compare that slice against the wider market."
)

if filtered_df.empty:
    st.warning("No rows match the selected filters.")
    st.stop()

salary_rows = salary_frame
classified_share = filtered_df["role_classified"].mean()
selected_postings = filtered_df["metadata_jobPostId"].nunique()
market_postings_total = market_view_df["metadata_jobPostId"].nunique()
scope_share = (
    selected_postings / market_postings_total
    if market_postings_total
    else float("nan")
)
median_salary = salary_rows[SALARY_COL].median() if not salary_rows.empty else float("nan")
market_salary = market_salary_frame[SALARY_COL].median() if not market_salary_frame.empty else float("nan")
mean_apv = filtered_df["applications_per_vacancy"].mean()
market_apv = market_view_df["applications_per_vacancy"].mean()
mean_rate = filtered_df["application_rate"].mean()
market_rate = market_view_df["application_rate"].mean()
salary_label = f"S${median_salary:,.0f}" if pd.notna(median_salary) else "N/A"
lead_sector, lead_sector_share = sector_focus_text(filtered_df)
lead_job_family, lead_job_family_share = job_family_focus_text(filtered_df)
salary_change = scope_salary_change(salary_rows)
salary_trend_heading, salary_trend_caption = salary_trend_story(salary_change, salary_trend)
comparison_mode = filters_active and (
    bool(filters["title_query"])
    or len(filters["categories"]) == 1
    or len(filters["roles"]) == 1
    or len(filters["positions"]) == 1
    or len(filters["employment"]) == 1
    or len(filters["experience"]) == 1
)
sector_filter_active = len(filters["categories"]) > 0
single_sector_mode = len(filters["categories"]) == 1
selected_sector_name = filters["categories"][0] if single_sector_mode else None

coverage_monthly = (
    filtered_df.groupby("posting_month")["metadata_jobPostId"]
    .nunique()
    .reset_index(name="postings")
    .sort_values("posting_month")
)
coverage_cutoff_date = None
if not coverage_monthly.empty:
    coverage_threshold = coverage_monthly["postings"].max() * 0.25
    coverage_match = coverage_monthly[coverage_monthly["postings"] >= coverage_threshold]
    if not coverage_match.empty:
        coverage_cutoff_date = coverage_match["posting_month"].min()

response_label = response_vs_market_label(mean_apv, mean_rate, market_apv, market_rate)
pay_delta = None if pd.isna(median_salary) or pd.isna(market_salary) or market_salary == 0 else (median_salary - market_salary) / market_salary
sector_median_apv = (
    market_bench["mean_applications_per_vacancy"].median()
    if not market_bench.empty
    else float("nan")
)
sector_median_salary = (
    market_bench["median_salary"].median()
    if not market_bench.empty
    else float("nan")
)
if single_sector_mode and selected_sector_name in market_bench["main_category"].values:
    selected_tightness_score = float(
        market_bench.loc[
            market_bench["main_category"] == selected_sector_name, "tightness_score"
        ].iloc[0]
    )
else:
    selected_tightness_score = scope_tightness_score(
        median_salary, mean_apv, mean_rate, market_bench
    )
selected_tightness_rank = (
    1 + int((market_bench["tightness_score"] > selected_tightness_score).sum())
    if not market_bench.empty and selected_tightness_score is not None
    else None
)
pay_change_text = ""
if salary_change and salary_change["pct_change"] is not None:
    direction = "up" if salary_change["pct_change"] > 0 else "down"
    pay_change_text = (
        f" Median pay is **{direction} {abs(salary_change['pct_change']):.1%}** "
        f"from **{money_md(salary_change['first_value'])}** in **{salary_change['first_month']:%b %Y}** "
        f"to **{money_md(salary_change['last_value'])}** in **{salary_change['last_month']:%b %Y}**."
    )

market_compare_text = (
    f" Median pay is **{money_md(median_salary)}** ({pay_delta:+.1%} vs the wider market in this period)."
    if pay_delta is not None
    else f" Median pay is **{money_md(median_salary)}**."
)

summary_text = (
    f"Across **{scope_label}**, hiring conditions are **{response_label}** "
    f"with **{mean_apv:.2f} applications per vacancy** versus **{market_apv:.2f}** for the wider market, "
    f"and an **{mean_rate:.1%} application rate** versus the market's **{market_rate:.1%}**. "
    f"This slice represents **{scope_share:.1%}** of captured postings in the selected period."
    f"{market_compare_text}{pay_change_text}"
)

st.markdown(filter_badges)
st.markdown(summary_text)

with st.container(horizontal=True):
    st.metric("Postings in scope", f"{selected_postings:,}", border=True)
    st.metric("Hiring companies", f"{filtered_df['postedCompany_name'].nunique():,}", border=True)
    st.metric(
        "Median pay",
        salary_label,
        delta=f"{median_salary - market_salary:+,.0f} vs wider market"
        if pd.notna(median_salary) and pd.notna(market_salary)
        else None,
        border=True,
    )
    st.metric(
        "Applications per vacancy",
        f"{mean_apv:.2f}",
        delta=f"{mean_apv - market_apv:+.2f} vs wider market",
        border=True,
    )
    st.metric(
        "Application rate",
        f"{mean_rate:.1%}",
        delta=f"{mean_rate - market_rate:+.1%} vs wider market",
        border=True,
    )
    st.metric(
        "Share of market",
        f"{scope_share:.0%}",
        delta=f"{selected_postings:,} of {market_postings_total:,} postings",
        border=True,
    )

with st.expander("How to read the hiring competition metrics", icon=":material/help:"):
    st.markdown(
        """
        - **Applications per vacancy** = total applications divided by the number of vacancies in each posting.
        - **Application rate** = total applications divided by total views.
        - In this dataset, **higher values mean stronger candidate response**, which usually suggests it is easier to attract applicants.
        - We do **not** use median repost count or days-open as the main hiring benchmark because repost data is too zero-heavy and posting duration is largely driven by platform rules.
        """
    )

tab1, tab2, tab3, tab4 = st.tabs(
    ["Where demand sits", "What to budget", "Where hiring looks tighter", "Filtered data"]
)

with tab1:
    c1, c2 = st.columns(2)

    demand_by_sector = (
        filtered_df.groupby("main_category")["metadata_jobPostId"]
        .nunique()
        .reset_index(name="postings")
        .sort_values("postings", ascending=False)
        .head(10)
    )

    with c1:
        if demand_by_sector.empty:
            st.subheader("Sector demand overview")
            st.caption("No sector demand view is available for the current filters.")
        elif single_sector_mode and comparison_mode:
            st.subheader(
                f"{lead_sector} represents {scope_share:.1%} of captured hiring demand in this period"
            )
            sector_share_chart = share_donut_chart(
                selected_postings,
                market_postings_total,
                lead_sector,
            )
            st.altair_chart(sector_share_chart)
            st.caption(
                f"The selected sector contributes **{selected_postings:,} postings** out of **{market_postings_total:,}** captured across the wider market in the same date window."
            )
        else:
            demand_leader = demand_by_sector.iloc[0]
            if lead_sector_share >= 0.60:
                st.subheader(
                    f"{scope_label.capitalize()} are concentrated in {demand_leader['main_category']}"
                )
            else:
                st.subheader(
                    f"{demand_leader['main_category']} leads demand in the current scope"
                )
            sector_chart = top_highlight_chart(
                demand_by_sector,
                "main_category",
                "postings",
                demand_leader["main_category"],
                "Unique postings",
            )
            st.altair_chart(sector_chart)
            st.caption(
                f"The highlighted sector accounts for **{lead_sector_share:.0%}** of postings in the current scope."
            )

    demand_over_time = (
        filtered_df.groupby("posting_month")["metadata_jobPostId"]
        .nunique()
        .reset_index(name="postings")
        .sort_values("posting_month")
    )

    with c2:
        if sector_filter_active:
            st.subheader(posting_trend_story(demand_over_time, scope_label.capitalize()))
            peer_trend_data = build_peer_trend_data(
                market_view_df,
                filtered_df,
                scope_label,
                filters["categories"],
                top_n=8,
            )
            series_order = [scope_label] + [
                name for name in peer_trend_data["series_label"].unique().tolist()
                if name != scope_label
            ]
            series_colors = [
                "#c2410c",
                "#0f766e",
                "#1d4ed8",
                "#7c3aed",
                "#475569",
                "#0284c7",
                "#9333ea",
                "#059669",
                "#64748b",
            ][: len(series_order)]
            trend_chart = (
                alt.Chart(peer_trend_data)
                .mark_line(point=True)
                .encode(
                    x=alt.X("posting_month:T", title="Posting month"),
                    y=alt.Y("postings:Q", title="Unique postings"),
                    detail="series_label:N",
                    color=alt.Color(
                        "series_label:N",
                        scale=alt.Scale(
                            domain=series_order,
                            range=series_colors,
                        ),
                        legend=alt.Legend(title=None),
                    ),
                    size=alt.condition(
                        alt.datum.highlight_group == "Selected scope",
                        alt.value(4),
                        alt.value(2),
                    ),
                    opacity=alt.condition(
                        alt.datum.highlight_group == "Selected scope",
                        alt.value(1.0),
                        alt.value(0.72),
                    ),
                    tooltip=["posting_month:T", "series_label:N", "postings:Q"],
                )
            )
        else:
            st.subheader(posting_trend_story(demand_over_time, scope_label.capitalize()))
            trend_base = alt.Chart(demand_over_time).encode(
                x=alt.X("posting_month:T", title="Posting month"),
                y=alt.Y("postings:Q", title="Unique postings"),
                tooltip=["posting_month:T", "postings"],
            )
            trend_chart = trend_base.mark_line(point=True, color="#0f766e")
        layers = []
        if coverage_cutoff_date is not None:
            shaded = alt.Chart(
                pd.DataFrame(
                    {
                        "start": [demand_over_time["posting_month"].min()],
                        "end": [coverage_cutoff_date],
                    }
                )
            ).mark_rect(color="#e2e8f0", opacity=0.65).encode(
                x="start:T",
                x2="end:T",
            )
            layers.append(shaded)
        layers.append(trend_chart)
        st.altair_chart(alt.layer(*layers).properties(height=360))
        if coverage_cutoff_date is not None:
            if sector_filter_active:
                st.caption(
                    f"The orange line is the selected scope and the other lines are the biggest peer sectors. Months before **{coverage_cutoff_date:%B %Y}** are shaded because posting volume is much lower and likely reflects weaker dataset collection coverage."
                )
            else:
                st.caption(
                    f"Months before **{coverage_cutoff_date:%B %Y}** are shaded because posting volume is much lower and likely reflects weaker dataset collection coverage."
                )
        else:
            if sector_filter_active:
                st.caption(
                    "The orange line is the selected scope and the other lines are the biggest peer sectors. Read this directionally: it shows the hiring volume captured by this dataset, not the full economy."
                )
            else:
                st.caption(
                    "Read this directionally: it shows the hiring volume captured by this dataset, not the full economy."
                )

    d1, d2 = st.columns(2)

    experience_mix = (
        filtered_df.groupby("experience_group")["metadata_jobPostId"]
        .nunique()
        .reindex(EXPERIENCE_ORDER)
        .reset_index(name="postings")
    )
    experience_compare = experience_mix_comparison(filtered_df, market_view_df)

    with d1:
        if filters_active:
            st.subheader(experience_mix_title(experience_compare))
            exp_chart = experience_mix_dumbbell_chart(experience_compare)
            st.altair_chart(exp_chart)
            st.caption(
                "This shows whether the selected hiring scope is more junior or more senior than the whole market."
            )
        else:
            st.subheader("Mid-level experience bands carry most of the market's hiring volume")
            exp_chart = (
                alt.Chart(experience_mix.dropna())
                .mark_bar(color="#7c3aed")
                .encode(
                    x=alt.X("experience_group:N", sort=EXPERIENCE_ORDER, title="Experience band"),
                    y=alt.Y("postings:Q", title="Unique postings"),
                    tooltip=["experience_group", "postings"],
                )
                .properties(height=320)
            )
            st.altair_chart(exp_chart)
            st.caption("This helps show where overall demand is concentrated across experience bands.")

    role_mix = (
        filtered_df[filtered_df["role_classified"]]
        .groupby("primary_role")["metadata_jobPostId"]
        .nunique()
        .reset_index(name="postings")
        .sort_values("postings", ascending=False)
        .head(10)
    )

    with d2:
        if role_mix.empty:
            st.subheader("Job-family breakdown in the current scope")
        elif lead_job_family_share >= 0.60:
            st.subheader(
                f"{lead_job_family} dominates the job-family mix in this view"
            )
        else:
            st.subheader("Job-family demand is spread across several role types")
        if role_mix.empty:
            st.caption("No job-family view is available in the current filters.")
        else:
            role_leader = role_mix.iloc[0]
            role_chart = top_highlight_chart(
                role_mix,
                "primary_role",
                "postings",
                role_leader["primary_role"],
                "Unique postings",
                height=320,
            )
            st.altair_chart(role_chart)
            st.caption(
                f"Here, **sector** means the employer's industry, while **job family** means the type of role being hired, such as software, finance, or operations. "
                f"About **{classified_share:.0%}** of postings in this view have a usable job-family label, and **{role_leader['primary_role']}** makes up **{lead_job_family_share:.0%}** of the classified mix."
            )

with tab2:
    s1, s2 = st.columns(2)

    if filters_active:
        pay_experience_options = ["All experience bands"] + [
            band
            for band in EXPERIENCE_ORDER
            if band in market_salary_frame["experience_group"].dropna().unique().tolist()
        ]
        if len(filters["experience"]) == 1:
            selected_pay_experience = filters["experience"][0]
            st.caption(f"Pay comparison is locked to **{selected_pay_experience}** because that experience band is already selected in the sidebar.")
        else:
            selected_pay_experience = st.selectbox(
            "Compare sector pay for",
                pay_experience_options,
                key="pay_experience_compare",
            )
    else:
        selected_pay_experience = "All experience bands"

    pay_salary_rows = (
        salary_rows
        if selected_pay_experience == "All experience bands"
        else salary_rows[salary_rows["experience_group"] == selected_pay_experience]
    )
    pay_market_salary_frame = (
        market_salary_frame
        if selected_pay_experience == "All experience bands"
        else market_salary_frame[
            market_salary_frame["experience_group"] == selected_pay_experience
        ]
    )
    pay_scope_salary = (
        pay_salary_rows[SALARY_COL].median() if not pay_salary_rows.empty else float("nan")
    )
    pay_market_median_salary = (
        pay_market_salary_frame[SALARY_COL].median()
        if not pay_market_salary_frame.empty
        else float("nan")
    )

    sector_salary = (
        pay_market_salary_frame.groupby("main_category")[SALARY_COL]
        .agg(median_salary="median", postings="size")
        .reset_index()
    )
    sector_salary = sector_salary[sector_salary["postings"] >= min_n].sort_values(
        "median_salary", ascending=False
    )

    with s1:
        if comparison_mode and sector_filter_active:
            st.subheader(
                f"{scope_label.capitalize()} pay {money_md(pay_scope_salary)} vs a sector median of {money_md(sector_salary['median_salary'].median() if not sector_salary.empty else float('nan'))}"
            )
        elif comparison_mode:
            st.subheader(
                benchmark_title(
                    scope_label,
                    pay_scope_salary,
                    pay_market_median_salary,
                    "pay",
                    money_metric=True,
                )
            )
        elif sector_salary.empty:
            st.subheader("Sector pay benchmark")
        else:
            salary_leader = sector_salary.iloc[0]
            st.subheader(
                f"{salary_leader['main_category']} has the highest median pay in this view"
            )
        if comparison_mode and sector_filter_active:
            pay_benchmark_chart = peer_sector_comparison_chart(
                sector_salary.rename(columns={"main_category": "main_category"}),
                scope_label,
                pay_scope_salary,
                "median_salary",
                "Median monthly salary (S$)",
                highlight_sector=selected_sector_name,
                ascending=False,
            )
            st.altair_chart(pay_benchmark_chart)
            st.caption(
                f"The orange marker is the selected scope. The dashed line is the median sector pay benchmark across the wider market for the same period"
                f"{'' if selected_pay_experience == 'All experience bands' else f' for {selected_pay_experience} roles'}."
            )
        elif comparison_mode:
            pay_benchmark_chart = benchmark_bar_chart(
                scope_label,
                pay_scope_salary,
                pay_market_median_salary,
                "Median monthly salary (S$)",
                height=260,
            )
            st.altair_chart(pay_benchmark_chart)
            if salary_change and salary_change["pct_change"] is not None:
                st.caption(
                    f"The selected scope pays **{money_md(pay_scope_salary)}** versus **{money_md(pay_market_median_salary)}** for the wider market in the same period"
                    f"{'' if selected_pay_experience == 'All experience bands' else f' for {selected_pay_experience} roles'}."
                )
            else:
                st.caption(
                    f"The selected scope pays **{money_md(pay_scope_salary)}** versus **{money_md(pay_market_median_salary)}** for the wider market in the same period"
                    f"{'' if selected_pay_experience == 'All experience bands' else f' for {selected_pay_experience} roles'}."
                )
        elif sector_salary.empty:
            st.caption("No sector has enough salary-clean rows for a stable benchmark.")
        else:
            top_salary_chart = sector_salary.head(10).copy()
            sal_chart = top_highlight_chart(
                top_salary_chart,
                "main_category",
                "median_salary",
                salary_leader["main_category"],
                "Median monthly salary (S$)",
            )
            st.altair_chart(sal_chart)
            st.caption(
                f"The highlighted bar shows the most expensive sector in this view at **{money_md(salary_leader['median_salary'])}** median monthly pay."
            )

    with s2:
        single_experience_selected = len(filters["experience"]) == 1
        if salary_exp.empty:
            st.subheader("Experience-band pay and demand")
        elif single_experience_selected:
            st.subheader(
                f"The selected view is already focused on {filters['experience'][0]} roles"
            )
        else:
            st.subheader(experience_pay_story(salary_exp))
        if salary_exp.empty:
            st.caption("No salary-clean rows are available in this view.")
        elif single_experience_selected:
            st.caption(
                "Because one experience band is already selected in the sidebar, the sector pay comparison and the salary trend below are the more useful apples-to-apples views here."
            )
        else:
            exp_chart_data = salary_exp.dropna().copy()
            bars = (
                alt.Chart(exp_chart_data)
                .mark_bar(color="#cbd5e1")
                .encode(
                    x=alt.X("experience_group:N", sort=EXPERIENCE_ORDER, title="Experience band"),
                    y=alt.Y("postings:Q", title="Salary-clean postings"),
                    tooltip=["experience_group", "median_salary", "postings"],
                )
            )
            line = (
                alt.Chart(exp_chart_data)
                .mark_line(point=True, color="#2563eb")
                .encode(
                    x=alt.X("experience_group:N", sort=EXPERIENCE_ORDER, title="Experience band"),
                    y=alt.Y("median_salary:Q", title="Median monthly salary (S$)"),
                    tooltip=["experience_group", "median_salary", "postings"],
                )
            )
            st.altair_chart(
                alt.layer(bars, line).resolve_scale(y="independent").properties(height=360)
            )
            st.caption(
                "Bars show where the hiring volume sits, while the line shows how pay steps up across experience bands."
            )

    selected_experience_band = (
        filters["experience"][0] if len(filters["experience"]) == 1 else None
    )
    if sector_filter_active and selected_experience_band is not None:
        st.subheader(
            f"{scope_label.capitalize()} {selected_experience_band} pay compared with peer sectors"
        )
        sector_salary_trend = build_sector_salary_trend_data(
            pay_market_salary_frame,
            pay_salary_rows,
            scope_label,
            filters["categories"],
            selected_experience_band,
            top_n=8,
        )
        if sector_salary_trend.empty:
            st.caption("No salary trend is available in this view.")
        else:
            sector_series_order = [scope_label, "Sector median"] + [
                name
                for name in sector_salary_trend["series_label"].unique().tolist()
                if name not in [scope_label, "Sector median"]
            ]
            sector_color_map = {
                scope_label: "#c2410c",
                "Sector median": "#0f172a",
            }
            peer_palette = [
                "#0f766e",
                "#1d4ed8",
                "#7c3aed",
                "#64748b",
                "#0284c7",
                "#059669",
                "#9333ea",
                "#475569",
            ]
            for name, color in zip(
                [n for n in sector_series_order if n not in sector_color_map],
                peer_palette,
            ):
                sector_color_map[name] = color
            sector_salary_chart = (
                alt.Chart(sector_salary_trend)
                .mark_line(point=True)
                .encode(
                    x=alt.X("posting_month:T", title="Posting month"),
                    y=alt.Y("median_salary:Q", title="Median monthly salary (S$)"),
                    detail="series_label:N",
                    color=alt.Color(
                        "series_label:N",
                        scale=alt.Scale(
                            domain=sector_series_order,
                            range=[sector_color_map[name] for name in sector_series_order],
                        ),
                        legend=alt.Legend(title=None),
                    ),
                    size=alt.Size("line_size:Q", legend=None),
                    opacity=alt.Opacity("line_opacity:Q", legend=None),
                    tooltip=["posting_month:T", "series_label:N", "median_salary:Q"],
                )
                .properties(height=380)
            )
            st.altair_chart(sector_salary_chart)
            st.caption(
                f"This compares **{selected_experience_band}** pay in the selected scope against the median for that same experience band and the largest peer sectors."
            )
    elif salary_trend.empty:
        st.subheader(salary_trend_heading)
        st.caption("No salary trend is available in this view.")
    else:
        st.subheader(salary_trend_heading)
        trend_salary_chart = (
            alt.Chart(salary_trend.dropna())
            .mark_line(point=True)
            .encode(
                x=alt.X("posting_month:T", title="Posting month"),
                y=alt.Y("median_salary:Q", title="Median monthly salary (S$)"),
                color=alt.Color("experience_group:N", sort=EXPERIENCE_ORDER, title="Experience band"),
                tooltip=["posting_month:T", "experience_group", "median_salary"],
            )
            .properties(height=380)
        )
        st.altair_chart(trend_salary_chart)
        st.caption(salary_trend_caption)

with tab3:
    t1, t2 = st.columns(2)

    with t1:
        selected_sector_row = None

        if market_bench.empty:
            st.subheader("Sector hiring map")
            st.caption("Not enough stable sector data is available for a hiring benchmark.")
        elif comparison_mode and sector_filter_active:
            st.subheader(
                f"{scope_label.capitalize()} receive {mean_apv:.2f} applications per vacancy vs a sector median of {sector_median_apv:.2f}"
            )
            apv_benchmark_chart = peer_sector_comparison_chart(
                market_bench,
                scope_label,
                mean_apv,
                "mean_applications_per_vacancy",
                "Applications per vacancy",
                highlight_sector=selected_sector_name,
                ascending=False,
            )
            st.altair_chart(apv_benchmark_chart)
            st.caption(
                "The orange marker is the selected scope. The dashed line is the median sector benchmark, and the grey markers are the eight largest peer sectors in the same period."
            )
        else:
            selected_sector_name = st.selectbox(
                "Highlight a sector on the map",
                market_bench["main_category"].tolist(),
                index=0,
            )
            selected_sector_row = market_bench[
                market_bench["main_category"] == selected_sector_name
            ].iloc[0]
            st.subheader(selected_sector_title(selected_sector_row, market_bench))

            x_rule = pd.DataFrame({"median_salary": [market_bench["median_salary"].median()]})
            y_rule = pd.DataFrame(
                {
                    "mean_applications_per_vacancy": [
                        market_bench["mean_applications_per_vacancy"].median()
                    ]
                }
            )

            chart_data = market_bench.copy()
            chart_data["highlight_group"] = chart_data["main_category"].apply(
                lambda x: "Highlighted sector" if x == selected_sector_name else "Other sectors"
            )

            points = (
                alt.Chart(chart_data)
                .mark_circle(size=180, opacity=0.85)
                .encode(
                    x=alt.X("median_salary:Q", title="Median monthly salary (S$)"),
                    y=alt.Y(
                        "mean_applications_per_vacancy:Q",
                        title="Average applications per vacancy",
                    ),
                    color=alt.Color(
                        "highlight_group:N",
                        scale=alt.Scale(
                            domain=["Highlighted sector", "Other sectors"],
                            range=["#c2410c", "#cbd5e1"],
                        ),
                        legend=None,
                    ),
                    size=alt.Size("postings:Q", title="Postings"),
                    tooltip=[
                        "main_category",
                        "postings",
                        "median_salary",
                        "mean_applications_per_vacancy",
                        "mean_application_rate",
                        "tightness_label",
                    ],
                )
            )

            vline = alt.Chart(x_rule).mark_rule(strokeDash=[6, 4], color="gray").encode(
                x="median_salary:Q"
            )
            hline = alt.Chart(y_rule).mark_rule(strokeDash=[6, 4], color="gray").encode(
                y="mean_applications_per_vacancy:Q"
            )

            st.altair_chart((points + vline + hline).properties(height=380))
            st.caption(
                f"{selected_sector_name} offers median pay of **{money_md(selected_sector_row['median_salary'])}** "
                f"and attracts **{selected_sector_row['mean_applications_per_vacancy']:.2f} applications per vacancy**. "
                "Use the selector and hover to inspect each sector. Sectors that sit lower and further right are usually tougher for employers."
            )

    with t2:
        if market_bench.empty:
            st.subheader("Sector tightness ranking")
        elif comparison_mode and sector_filter_active:
            st.subheader(
                f"{scope_label.capitalize()} rank {ordinal_text(selected_tightness_rank)} tightest out of {len(market_bench)} sectors"
            )
        else:
            tightest_sector = market_bench.iloc[0]
            st.subheader(
                f"{tightest_sector['main_category']} ranks as the hardest sector to hire for in this view"
            )
        if market_bench.empty:
            st.caption("No stable tightness benchmark is available.")
        elif comparison_mode and sector_filter_active:
            rate_benchmark_chart = tightness_rank_chart(
                market_bench,
                highlight_sector=selected_sector_name,
                selected_score=selected_tightness_score,
                selected_label=scope_label,
                top_n=8,
                height=320,
            )
            st.altair_chart(rate_benchmark_chart)
            st.caption(
                f"Hiring tightness is a weighted score built from **lower applications per vacancy (45%)**, **lower application rate (35%)**, and **higher pay (20%)**. "
                f"Higher scores mean the sector appears harder for employers to hire into. The selected scope scores **{selected_tightness_score:.2f}**."
                if selected_tightness_score is not None
                else "Higher scores mean the sector appears harder for employers to hire into."
            )
        else:
            st.altair_chart(tightness_rank_chart(market_bench))
            st.caption(
                f"The current tightest sector is **{tightest_sector['main_category']}** with "
                f"median pay of **{money_md(tightest_sector['median_salary'])}**, "
                f"**{tightest_sector['mean_applications_per_vacancy']:.2f} applications per vacancy**, "
                f"and an average application rate of **{tightest_sector['mean_application_rate']:.1%}**. "
                "Hiring tightness is a weighted score built from **lower applications per vacancy (45%)**, "
                "**lower application rate (35%)**, and **higher pay (20%)**."
            )

    if single_sector_mode and not seniority_bench.empty:
        toughest_band = seniority_bench.sort_values(
            "mean_applications_per_vacancy", ascending=True
        ).iloc[0]
        strongest_band = seniority_bench.sort_values(
            "mean_applications_per_vacancy", ascending=False
        ).iloc[0]
        st.subheader(
            f"Inside {selected_sector_name}, {toughest_band['experience_group']} roles draw the fewest applicants"
        )
    elif role_bench.empty:
        st.subheader("Role-family response benchmark")
    elif comparison_mode and len(role_bench) <= 1:
        st.subheader("Job-family response sits close to the selected scope average")
    else:
        weakest_role = role_bench.iloc[0]
        st.subheader(
            f"{weakest_role['primary_role']} draws only {weakest_role['mean_applications_per_vacancy']:.2f} applications per vacancy"
        )
    if single_sector_mode and not seniority_bench.empty:
        seniority_chart_data = seniority_bench.dropna().copy()
        key_bands = {
            toughest_band["experience_group"],
            strongest_band["experience_group"],
        }
        seniority_chart_data["highlight_group"] = seniority_chart_data[
            "experience_group"
        ].apply(lambda x: "Key bands" if x in key_bands else "Other bands")
        seniority_chart = (
            alt.Chart(seniority_chart_data)
            .mark_bar(cornerRadiusEnd=5)
            .encode(
                x=alt.X(
                    "mean_applications_per_vacancy:Q",
                    title="Average applications per vacancy",
                ),
                y=alt.Y(
                    "experience_group:N",
                    sort=EXPERIENCE_ORDER,
                    title=None,
                ),
                color=alt.Color(
                    "highlight_group:N",
                    scale=alt.Scale(
                        domain=["Key bands", "Other bands"],
                        range=["#c2410c", "#cbd5e1"],
                    ),
                    legend=None,
                ),
                tooltip=[
                    "experience_group",
                    "postings",
                    alt.Tooltip("mean_applications_per_vacancy:Q", format=".2f"),
                    alt.Tooltip("mean_application_rate:Q", format=".1%"),
                ],
            )
            .properties(height=300)
        )
        st.altair_chart(seniority_chart)
        st.caption(
            f"Within **{selected_sector_name}**, **{strongest_band['experience_group']}** roles attract **{strongest_band['mean_applications_per_vacancy']:.2f} applications per vacancy**, "
            f"versus only **{toughest_band['mean_applications_per_vacancy']:.2f}** for **{toughest_band['experience_group']}** roles. "
            "That suggests applicant depth is much stronger in some seniority bands than others within the same sector."
        )
    elif role_bench.empty:
        st.caption("No job-family benchmark is available for this view.")
    elif comparison_mode and len(role_bench) <= 1:
        st.caption(
            "The selected scope is already narrow enough that an internal job-family comparison is not very informative, so the whole-market benchmarks above are more useful."
        )
    else:
        weakest_roles = role_bench.head(10).copy()
        weakest_roles["highlight_group"] = weakest_roles["primary_role"].apply(
            lambda x: "Weakest response" if x == weakest_role["primary_role"] else "Other roles"
        )
        weak_role_chart = (
            alt.Chart(weakest_roles)
            .mark_circle(size=180)
            .encode(
                x=alt.X(
                    "mean_applications_per_vacancy:Q",
                    title="Average applications per vacancy",
                ),
                y=alt.Y("primary_role:N", sort="x", title=None),
                color=alt.Color(
                    "highlight_group:N",
                    scale=alt.Scale(
                        domain=["Weakest response", "Other roles"],
                        range=["#b91c1c", "#cbd5e1"],
                    ),
                    legend=None,
                ),
                tooltip=[
                    "primary_role",
                    "postings",
                    "mean_applications_per_vacancy",
                    "mean_application_rate",
                ],
            )
            .properties(height=340)
        )
        st.altair_chart(weak_role_chart)
        st.caption(
            f"**{weakest_role['primary_role']}** attracts only **{weakest_role['mean_applications_per_vacancy']:.2f} applications per vacancy**, "
            f"which is well below the filtered-view average of **{mean_apv:.2f}**. "
            "This chart excludes `Other / Unclassified` so the comparison stays usable."
        )

with tab4:
    st.subheader("Filtered data preview")
    show_cols = [
        "title",
        "postedCompany_name",
        "main_category",
        "primary_role",
        "positionLevels",
        "employmentTypes",
        "minimumYearsExperience",
        "experience_group",
        "salary_min_clean",
        "salary_max_clean",
        SALARY_COL,
        "metadata_totalNumberJobApplication",
        "numberOfVacancies",
        "applications_per_vacancy",
        "metadata_totalNumberOfView",
        "application_rate",
        "metadata_originalPostingDate",
    ]
    show_cols = [col for col in show_cols if col in filtered_df.columns]

    st.dataframe(
        filtered_df[show_cols].head(1000),
        hide_index=True,
        column_config={
            "salary_min_clean": st.column_config.NumberColumn("Salary min", format="S$ %.0f"),
            "salary_max_clean": st.column_config.NumberColumn("Salary max", format="S$ %.0f"),
            SALARY_COL: st.column_config.NumberColumn("Average salary", format="S$ %.0f"),
            "applications_per_vacancy": st.column_config.NumberColumn(
                "Apps / vacancy", format="%.2f"
            ),
            "application_rate": st.column_config.NumberColumn(
                "Application rate", format="percent"
            ),
            "metadata_originalPostingDate": st.column_config.DateColumn(
                "Posting date", format="YYYY-MM-DD"
            ),
        },
    )
    st.caption("Showing the first 1,000 filtered rows.")

    st.download_button(
        "Download filtered rows as CSV",
        filtered_df[show_cols].to_csv(index=False).encode("utf-8"),
        file_name="filtered_jobs.csv",
        mime="text/csv",
    )
